# Asisten AI Legal berbasis RAG — Regulasi Cipta Kerja

Notebook ini membangun **pipeline Retrieval-Augmented Generation (RAG)** modular untuk
asisten hukum tim legal internal. Sistem menjawab pertanyaan seputar **empat regulasi
turunan Undang-Undang Cipta Kerja** dengan jawaban yang **di-grounding ke pasal sumber**,
bukan halusinasi model.

**Empat dokumen sumber pengetahuan:**

| Kode | Regulasi | Tentang |
|---|---|---|
| `PP_5_2021` | PP No. 5 Tahun 2021 | Perizinan Berusaha Berbasis Risiko |
| `PP_35_2021` | PP No. 35 Tahun 2021 | PKWT, Alih Daya, Waktu Kerja, PHK |
| `PP_51_2023` | PP No. 51 Tahun 2023 | Pengupahan (formula upah minimum) |
| `UU_6_2023` | UU No. 6 Tahun 2023 | Cipta Kerja (omnibus, 15 BAB) |

**Desain kunci:** pengetahuan hukum berasal dari **retrieval** (4 PDF), bukan dari bobot model.
Model generator (`Llama-3.2-3B` yang telah di-fine-tune) berperan menyusun jawaban Bahasa
Indonesia formal berdasarkan konteks yang di-retrieve.

**Arsitektur modular (satu layer per tanggung jawab, mudah di-swap ke domain lain):**

```
PDF → Loader → Cleaner → Chunker → Embedder (BGE-M3) → Vector Store (ChromaDB)
    → Retriever → Generator (fine-tuned Llama)
```

Bagian **A (RAG Dasar)** di notebook ini mencakup pipeline end-to-end pertama: memuat 4 PDF,
memecah menjadi chunk per-pasal dengan *overlap* eksplisit, meng-*embed* dengan BGE-M3,
menyimpan ke ChromaDB persisten (18 koleksi), lalu men-*generate* jawaban dengan model
fine-tuned.


## A.1 — Lingkungan & Dependensi

Jalankan sel instalasi di bawah. **Pada eksekusi pertama**, setelah instalasi selesai,
lakukan *Restart runtime* (Runtime → Restart session) lalu **Run All** ulang agar
seluruh library ter-*update* termuat bersih. Notebook dirancang untuk **Google Colab
GPU T4**.

In [1]:
# Instalasi dependensi RAG (embedder, vector store, retriever, reranker, generator, web-fallback)
%pip install -q chromadb sentence-transformers pypdf pyyaml rank-bm25 ddgs gradio
%pip install -q -U transformers accelerate bitsandbytes
!apt-get -qq install -y aria2 > /dev/null 2>&1
import os
# HF downloader bawaan kadang hang di file model besar (LFS) pada environment tertentu;
# unduhan model besar memakai aria2c (16 koneksi + resume) -> nonaktifkan hf_transfer.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
print(">> Dependensi terpasang. Jika ini eksekusi pertama: Restart runtime lalu Run All ulang.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 46.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3

In [2]:
# Import inti + reproducibility (seed=42 di semua operasi acak)
import os, re, json, random, time
from pathlib import Path
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Torch:", torch.__version__, "| CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.11.0+cu128 | CUDA tersedia: True
GPU: Tesla T4


**(Opsional) Autentikasi Hugging Face.** Semua model bersifat publik sehingga token
**tidak wajib**. Namun tanpa token, HF Hub membatasi rate download dan bisa lambat/*stall*.
Set Colab Secret bernama `HF_TOKEN` (role *Read*) untuk unduhan lebih cepat dan stabil.

In [3]:
# Autentikasi HF opsional -> download model lebih cepat/stabil (skip diam-diam kalau tak ada)
try:
    from google.colab import userdata
    _hf = userdata.get("HF_TOKEN")
except Exception:
    _hf = os.environ.get("HF_TOKEN")
if _hf:
    os.environ["HF_TOKEN"] = _hf
    try:
        from huggingface_hub import login
        login(_hf)
        print("HF login OK (unduhan lebih cepat & stabil).")
    except Exception as e:
        print("HF login dilewati:", e)
else:
    print("Tanpa HF_TOKEN (unduhan publik). Jika unduhan stall, set Colab Secret HF_TOKEN.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK (unduhan lebih cepat & stabil).


## A.2 — Konfigurasi (config-driven)

Seluruh *magic number* (ukuran chunk, overlap, top-k, ID model, ambang) dikumpulkan dalam
satu objek konfigurasi. Pendekatan ini membuat pipeline **portabel ke domain lain** —
cukup ganti PDF sumber dan nilai konfigurasi, tanpa menyentuh logika.

In [4]:
# Konfigurasi terpusat untuk pipeline RAG
RAG_CONFIG = {
    "seed": 42,
    "chunker": {
        "chunk_size": 1200,      # target char per chunk (batas rubric <= 5000)
        "chunk_overlap": 100,    # overlap EKSPLISIT antar sub-chunk (bukan default library)
        "hard_cap": 5000,        # batas keras char per chunk
    },
    "embedder": {
        "model_id": "BAAI/bge-m3",
        "batch_size": 32,
        "normalize": True,
    },
    "vector_store": {
        "distance_metric": "cosine",
    },
    "retriever": {
        "top_k": 5,              # jumlah chunk konteks untuk generator (tier Basic)
    },
    "generator": {
        "model_id": "nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO",  # hasil fine-tuning K1
        "max_new_tokens": 512,
        "temperature": 0.3,      # rendah -> jawaban legal deterministik
        "top_p": 0.9,
        "repetition_penalty": 1.1,
    },
    "system_prompt": (
        "Anda adalah asisten hukum untuk tim legal internal. Jawab HANYA berdasarkan "
        "konteks pasal yang diberikan. Gunakan Bahasa Indonesia hukum yang formal dan "
        "ringkas. Jika konteks tidak memuat jawaban, katakan: 'Informasi tidak ditemukan "
        "pada regulasi yang tersedia.'"
    ),
    "context_template": "Konteks pasal:\n{sources}\n\nPertanyaan: {question}\nJawaban:",
}
print("Konfigurasi RAG dimuat.")

Konfigurasi RAG dimuat.


## A.3 — Mount Google Drive & Path

Empat PDF sumber diletakkan di Drive: `/content/drive/MyDrive/PGABL/data/raw/`.
ChromaDB persisten disimpan ke Drive agar indeks bertahan lintas sesi.

In [5]:
# Mount Drive + definisi path
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/PGABL")
except ImportError:
    BASE = Path("./PGABL")   # fallback lokal

RAW_DIR = BASE / "data" / "raw"
CHROMA_DIR = str(BASE / "chroma_db")
(BASE / "data").mkdir(parents=True, exist_ok=True)

# Empat PDF: nama file -> kode sumber (path-safe)
PDF_FILES = {
    "PP_5_2021":  RAW_DIR / "PP_5_2021.pdf",
    "PP_35_2021": RAW_DIR / "PP_35_2021.pdf",
    "PP_51_2023": RAW_DIR / "PP_51_2023.pdf",
    "UU_6_2023":  RAW_DIR / "UU_6_2023.pdf",
}
for k, p in PDF_FILES.items():
    print(f"{k:12s}: {'OK ' if p.exists() else 'HILANG'} {p}")
assert all(p.exists() for p in PDF_FILES.values()), (
    "Sebagian PDF belum ada. Unggah 4 PDF ke /content/drive/MyDrive/PGABL/data/raw/ "
    "dengan nama: PP_5_2021.pdf, PP_35_2021.pdf, PP_51_2023.pdf, UU_6_2023.pdf")

Mounted at /content/drive
PP_5_2021   : OK  /content/drive/MyDrive/PGABL/data/raw/PP_5_2021.pdf
PP_35_2021  : OK  /content/drive/MyDrive/PGABL/data/raw/PP_35_2021.pdf
PP_51_2023  : OK  /content/drive/MyDrive/PGABL/data/raw/PP_51_2023.pdf
UU_6_2023   : OK  /content/drive/MyDrive/PGABL/data/raw/UU_6_2023.pdf


## A.4 — Loader, Cleaner, dan Chunker

Tiga layer preprocessing (di-*inline* agar notebook mandiri):

- **Loader** — ekstrak teks per-halaman dengan `pypdf`.
- **Cleaner** — buang header/footer berulang (`PRESIDEN REPUBLIK INDONESIA`, nomor halaman),
  normalisasi artefak OCR (`2O21`→`2021`), rekonstruksi kata terpotong antar-halaman.
- **Chunker** — pecah **per-pasal**; satu pasal = satu chunk bila ≤ `chunk_size`, dan
  **pasal panjang di-sub-split dengan overlap eksplisit 100 karakter**. Untuk `UU_6_2023`,
  setiap chunk dipetakan ke salah satu dari **15 klaster (= 15 BAB omnibus)**; bagian
  Penjelasan di-route ke klaster berdasarkan konten.

In [6]:
# --- Loader ---
"""
PGABL - PDF Loader untuk RAG (Tahap 1b + Tahap 3).

Design:
- Default pakai pypdf (fastest, identical output vs pdfplumber untuk 4 PDF ini).
- Fallback pdfplumber tersedia via load_pdf(strategy="pdfplumber").
- Reusable ke domain lain: ganti file di data/raw/, tidak sentuh code.

Verify-first prototype: scripts/01_verify_pdf_loader.py
Hasil: semua 4 PDF (PP_5_2021, PP_35_2021, PP_51_2023, UU_6_2023) pakai pypdf clean.
"""

from __future__ import annotations
from pathlib import Path
from typing import Literal


LoaderStrategy = Literal["pypdf", "pdfplumber"]


def load_pdf_pages(pdf_path: Path, strategy: LoaderStrategy = "pypdf") -> list[str]:
    """
    Load semua halaman PDF menjadi list string per-halaman.

    Args:
        pdf_path: absolute atau relative path ke PDF
        strategy: "pypdf" (default, 2-4x lebih cepat) atau "pdfplumber" (robust untuk edge cases)

    Returns:
        List of strings, satu per halaman. String kosong kalau extract_text gagal.
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    if strategy == "pypdf":
        return _load_with_pypdf(pdf_path)
    elif strategy == "pdfplumber":
        return _load_with_pdfplumber(pdf_path)
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Use 'pypdf' or 'pdfplumber'.")


def _load_with_pypdf(pdf_path: Path) -> list[str]:
    import pypdf
    pages: list[str] = []
    with open(pdf_path, "rb") as f:
        reader = pypdf.PdfReader(f)
        for page in reader.pages:
            try:
                pages.append(page.extract_text() or "")
            except Exception as e:
                pages.append(f"[pypdf error: {e}]")
    return pages


def _load_with_pdfplumber(pdf_path: Path) -> list[str]:
    import pdfplumber
    pages: list[str] = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            try:
                pages.append(page.extract_text() or "")
            except Exception as e:
                pages.append(f"[pdfplumber error: {e}]")
    return pages

In [7]:
# --- Cleaner ---
"""
PGABL - Text Cleaner untuk PDF Legal Indonesia (Tahap 1b).

Kegunaan: strip repeated header/footer + normalize OCR artefak yang common
di dokumen regulasi Indonesia (PP, UU).

Design:
- Config regex di atas (HEADER_PATTERNS, FOOTER_PATTERNS, OCR_REPLACEMENTS).
- Ganti pattern untuk domain lain via extend config, jangan ubah logic.

Verify-first prototype: scripts/02_verify_chunker.py
Hasil: 4 PDF dgn cleanup delta -37 (PP_5_2021), -2052 (PP_35_2021),
       -590 (PP_51_2023), -23,746 (UU_6_2023).
"""

from __future__ import annotations
import re


# ==================== Header patterns (repeat tiap halaman) ====================
HEADER_PATTERNS = [
    # PRESIDEN\nREPUBLIK INDONESIA (dan variant garbled: NEPUBUK, REPUBLTI(, REPUELIK, dst)
    r"PRESIDEN\s*\n\s*(?:REPUBLIK|NEPUBUK|REPUBLTI\(?|REPUBLTK|REPTIBLIK|REPUELIK)\s*INDONESIA\s*\n",
    # LEMBARAN NEGARA header (cover page only)
    r"LEMBARAN\s+NEGARA\s*\n\s*REPUBLIK\s+INDONESIA\s*\n",
    # SALINAN header (tepat sebelum PRESIDEN)
    r"^\s*SALINAN\s*\n",
]


# ==================== Footer patterns ====================
FOOTER_PATTERNS = [
    # Page number "-N-" (dgn spasi/tanpa)
    r"\n\s*-\s*\d+\s*-\s*\n",
    r"\n\s*-\s*\d+\s*-\s*$",
    # SK No XXXXX A footer (tapi bukan SK dalam text lain)
    r"SK\s+No\s*[\d\s]+\s*[A-Z]?\s*(?:\n|$)",
]


# ==================== OCR replacements ====================
# Format: {regex_pattern: replacement}
OCR_REPLACEMENTS = {
    # Garbled REPUBLIK variants
    r"\bNEPUBUK\b": "REPUBLIK",
    r"\bREPUBLTI\(?\b": "REPUBLIK",
    r"\bREPUBLTK\b": "REPUBLIK",
    r"\bREPTIBLIK\b": "REPUBLIK",
    r"\bREPUELIK\b": "REPUBLIK",
    # Lower-case 'l' bukannya 'I'
    r"\blndonesia\b": "Indonesia",
    r"\blndo\b": "Indo",
    # Common misspellings from OCR
    r"\bkemanusraan\b": "kemanusiaan",
    r"\bpersekutrran\b": "persekutuan",
}


def normalize_ocr(text: str) -> str:
    """Fix OCR artefak yang common di dokumen legal Indonesia."""
    # Digit O di dalam angka (2O21 -> 2021), tapi bukan huruf O di kata biasa
    text = re.sub(r"(\d)O(\d)", r"\g<1>0\g<2>", text)
    text = re.sub(r"(\d)O$", r"\g<1>0", text, flags=re.MULTILINE)
    # Digit l di dalam angka (l945 -> 1945)
    text = re.sub(r"(\d)l(\d)", r"\g<1>1\g<2>", text)
    text = re.sub(r"\bl(\d{3,4})\b", r"1\g<1>", text)
    # Word-level replacements
    for pattern, repl in OCR_REPLACEMENTS.items():
        text = re.sub(pattern, repl, text)
    return text


def strip_headers_footers(text: str) -> str:
    """Buang repeated header/footer patterns."""
    for pat in HEADER_PATTERNS:
        text = re.sub(pat, "", text, flags=re.MULTILINE | re.IGNORECASE)
    for pat in FOOTER_PATTERNS:
        text = re.sub(pat, "\n", text, flags=re.MULTILINE)
    return text


def reconstruct_hyphenated_words(pages: list[str]) -> list[str]:
    """
    Gabungkan kata yang terpotong antar halaman (mis. 'keman-\nusraan' -> 'kemanusiaan').
    In-place modification, tapi return new list.
    """
    result = list(pages)
    for i in range(len(result) - 1):
        current = result[i].rstrip()
        m = re.search(r"(\w+)-\s*$", current)
        if m:
            trailing = m.group(1)
            next_page = result[i + 1].lstrip()
            n = re.match(r"(\w+)(.*)", next_page, re.DOTALL)
            if n:
                first_word = n.group(1)
                rest = n.group(2)
                result[i] = current[: -(len(trailing) + 1)] + trailing + first_word
                result[i + 1] = rest
    return result


def clean_pages(pages: list[str]) -> list[str]:
    """
    Full pipeline: reconstruct hyphenated -> strip headers/footers -> normalize OCR.

    Return: list halaman baru dgn text bersih.
    """
    pages = reconstruct_hyphenated_words(pages)
    cleaned = []
    for p in pages:
        p = strip_headers_footers(p)
        p = normalize_ocr(p)
        cleaned.append(p)
    return cleaned

In [8]:
# --- Chunker (per-pasal + sub-split overlap + routing klaster UU) ---
"""
PGABL - Chunker untuk Legal PDF (Tahap 1b + Tahap 3).

Strategy per-pasal (flat) untuk semua 4 PDF:
- Regex `^\\s*Pasal\\s+(\\d+)\\s*$` split jadi 1 blok per pasal.
- Blok pasal > chunk_size di-sub-split dgn OVERLAP EKSPLISIT (rubric WAJIB).
- Metadata: bab (Roman numeral), pasal, chunk_id, pdf_source.
- Khusus UU_6_2023: map BAB Roman I-XV -> klaster_id 1-15 + label tema
  (dipakai untuk 15 collection ChromaDB per klaster di Tahap 3).

Verify-first prototype: scripts/09_full_ingest_rag.py -> outputs/samples/.

Reusable ke domain lain: ganti PASAL_REGEX / BAB_REGEX / klaster map via argumen,
logic split tidak berubah.
"""

from __future__ import annotations
import re
from typing import Optional


# Pasal detection - baris tersendiri, toleran spasi
PASAL_REGEX = re.compile(r"^\s*Pasal\s+(\d+)\s*$", re.MULTILINE | re.IGNORECASE)

# BAB detection - Roman numeral standalone
BAB_REGEX = re.compile(r"^\s*BAB\s+([IVXLCDM]+)\s*$", re.MULTILINE)

# "Cukup jelas" di section Penjelasan (low-signal, di-skip)
CUKUP_JELAS_REGEX = re.compile(
    r"Pasal\s+\d+\s*\n\s*Cukup\s+jelas\s*\.?\s*(?:\n|$)",
    re.IGNORECASE,
)


# ==================== UU 6/2023: BAB (Roman) -> Klaster ====================
# UU 6/2023 (Cipta Kerja) punya TEPAT 15 BAB. "15 klaster" di desain = 15 BAB.
# Terverifikasi via recon lokal (semua BAB I-XV terdeteksi bersih).
# Label tema dipakai untuk sitasi sumber yang enak dibaca legal team.
UU_6_2023_KLASTER_LABELS = {
    1: "Ketentuan Umum",
    2: "Asas, Tujuan, dan Ruang Lingkup",
    3: "Peningkatan Ekosistem Investasi dan Kegiatan Berusaha",
    4: "Ketenagakerjaan",
    5: "Koperasi dan UMK-M",
    6: "Kemudahan Berusaha",
    7: "Dukungan Riset dan Inovasi",
    8: "Pengadaan Tanah",
    9: "Kawasan Ekonomi",
    10: "Investasi Pemerintah Pusat dan Proyek Strategis Nasional",
    11: "Administrasi Pemerintahan",
    12: "Pengawasan dan Pembinaan",
    13: "Ketentuan Lain-Lain",
    14: "Ketentuan Peralihan",
    15: "Ketentuan Penutup",
}

_ROMAN_MAP = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}


# Keyword routing untuk chunk PENJELASAN (yang tak bisa di-track per-pasal karena didominasi
# referensi UU sektor). Chunk penjelasan diarahkan ke klaster dgn hit keyword terbanyak;
# kalau nol match -> fallback klaster 3 (Ekosistem Investasi = cluster dominan/catch-all).
# Keyword lowercase; dicek substring pada text lowercase.
UU_6_2023_KLASTER_KEYWORDS = {
    4: ["pekerja", "buruh", "pengupahan", "upah minimum", "upah kerja", "lembur",
        "pkwt", "pemutusan hubungan kerja", "pesangon", "tenaga kerja", "serikat pekerja",
        "waktu kerja", "hubungan kerja", "perjanjian kerja", "jaminan sosial",
        "tenaga kerja asing", "pelatihan kerja"],
    5: ["koperasi", "usaha mikro", "usaha kecil", "usaha menengah", "umk-m", "umkm"],
    6: ["perseroan terbatas", "badan hukum", "keimigrasian", "hak kekayaan intelektual",
        "paten", "merek", "perkumpulan"],
    8: ["pengadaan tanah", "ganti kerugian", "pelepasan hak", "bank tanah"],
    9: ["kawasan ekonomi khusus", "kawasan perdagangan bebas", "kawasan ekonomi",
        "zona ekonomi"],
    10: ["proyek strategis nasional", "investasi pemerintah pusat", "lembaga pengelola investasi"],
    7: ["riset dan inovasi", "riset", "inovasi"],
    3: ["perizinan berusaha", "nomor induk berusaha", "sistem oss", "kbli", "berbasis risiko",
        "lingkungan hidup", "bangunan gedung", "tata ruang", "kehutanan", "sumber daya air",
        "kelautan", "perikanan", "ketenaganukliran", "pelayaran", "penerbangan", "pos",
        "telekomunikasi", "energi", "ketenagalistrikan", "perdagangan", "perindustrian"],
}


def route_penjelasan_by_keyword(text: str, fallback_klaster: int = 3) -> int:
    """Klaster untuk 1 chunk penjelasan via hit keyword terbanyak; nol match -> fallback."""
    low = text.lower()
    best_k, best_hits = fallback_klaster, 0
    for kid, kws in UU_6_2023_KLASTER_KEYWORDS.items():
        hits = sum(low.count(kw) for kw in kws)
        if hits > best_hits:
            best_k, best_hits = kid, hits
    return best_k


def roman_to_int(roman: str) -> Optional[int]:
    """Konversi Roman numeral ke int. Return None kalau input invalid."""
    if not roman:
        return None
    roman = roman.upper()
    total, prev = 0, 0
    for ch in reversed(roman):
        if ch not in _ROMAN_MAP:
            return None
        val = _ROMAN_MAP[ch]
        total += -val if val < prev else val
        prev = max(prev, val)
    return total if total > 0 else None


def find_pasal_boundaries(full_text: str) -> list[tuple[int, str]]:
    """Return list of (char_offset, pasal_number)."""
    return [(m.start(), m.group(1)) for m in PASAL_REGEX.finditer(full_text)]


def find_bab_boundaries(full_text: str) -> list[tuple[int, str]]:
    """Return list of (char_offset, bab_id)."""
    return [(m.start(), m.group(1)) for m in BAB_REGEX.finditer(full_text)]


def which_bab_for_offset(offset: int, bab_boundaries: list[tuple[int, str]]) -> Optional[str]:
    """Return BAB yang mencakup offset ini, atau None kalau di luar BAB apapun."""
    current = None
    for pos, bab in bab_boundaries:
        if pos <= offset:
            current = bab
        else:
            break
    return current


def split_text_with_overlap(text: str, chunk_size: int, overlap: int) -> list[str]:
    """
    Sliding-window split dgn OVERLAP EKSPLISIT (rubric WAJIB, bukan default library).

    Kalau text <= chunk_size -> 1 chunk utuh (tidak dipecah).
    Kalau lebih panjang -> potongan sepanjang chunk_size dgn step (chunk_size - overlap).
    Invariant: part[k][-overlap:] == part[k+1][:overlap] (overlap persis `overlap` char).
    """
    if overlap >= chunk_size:
        raise ValueError(f"overlap ({overlap}) harus < chunk_size ({chunk_size})")
    if len(text) <= chunk_size:
        return [text]
    parts: list[str] = []
    step = chunk_size - overlap
    start = 0
    while start < len(text):
        parts.append(text[start:start + chunk_size])
        if start + chunk_size >= len(text):
            break
        start += step
    return parts


def chunk_by_pasal(
    full_text: str,
    pdf_source: str,
    include_bab_metadata: bool = True,
    chunk_size: int = 1200,
    chunk_overlap: int = 100,
    hard_cap: int = 5000,
    skip_cukup_jelas: bool = True,
    map_klaster_by_bab: bool = False,
) -> list[dict]:
    """
    Chunker flat per-pasal dgn sub-split overlap eksplisit.

    Args:
        full_text: teks lengkap PDF (setelah cleaner).
        pdf_source: nama PDF tanpa .pdf (mis. "PP_35_2021").
        include_bab_metadata: kalau True, tambah field 'bab' di setiap chunk.
        chunk_size: panjang target chunk (char). Blok pasal > ini di-sub-split.
        chunk_overlap: overlap antar sub-chunk (char). EKSPLISIT (rubric WAJIB).
        hard_cap: batas keras char per chunk (rubric max 5000) - safety net.
        skip_cukup_jelas: skip pasal Penjelasan yang isinya cuma "Cukup jelas".
        map_klaster_by_bab: kalau True (khusus UU_6_2023), tambah klaster_id + klaster_label
                            dari BAB Roman (I->1 .. XV->15).

    Returns:
        List dict, tiap chunk punya:
          - chunk_id: str unik (mis. "PP_35_2021_pasal_1_0" atau "..._0_sub1" utk sub-chunk)
          - pdf_source, pasal, bab (Optional)
          - text: str (isi, di-cap di hard_cap)
          - length: int (panjang blok pasal ORIGINAL, sebelum sub-split)
          - char_len: int (panjang text chunk ini)
          - is_subchunk: bool, sub_index: int, n_subchunks: int
          - (UU only) klaster_id: Optional[int], klaster_label: Optional[str]
    """
    pasal_boundaries = find_pasal_boundaries(full_text)
    bab_boundaries = find_bab_boundaries(full_text) if include_bab_metadata else []

    chunks: list[dict] = []
    for i, (start, pasal_num) in enumerate(pasal_boundaries):
        end = pasal_boundaries[i + 1][0] if i + 1 < len(pasal_boundaries) else len(full_text)
        block = full_text[start:end].strip()

        # Skip "Cukup jelas" pasal (low-signal dari Penjelasan)
        if skip_cukup_jelas and CUKUP_JELAS_REGEX.search(block) and len(block) < 100:
            continue

        bab = which_bab_for_offset(start, bab_boundaries) if include_bab_metadata else None
        klaster_id = None
        klaster_label = None
        if map_klaster_by_bab and bab is not None:
            kid = roman_to_int(bab)
            if kid is not None and kid in UU_6_2023_KLASTER_LABELS:
                klaster_id = kid
                klaster_label = UU_6_2023_KLASTER_LABELS[kid]

        sub_texts = split_text_with_overlap(block, chunk_size, chunk_overlap)
        n_sub = len(sub_texts)
        for j, sub in enumerate(sub_texts):
            chunk_id = (
                f"{pdf_source}_pasal_{pasal_num}_{i}"
                if n_sub == 1
                else f"{pdf_source}_pasal_{pasal_num}_{i}_sub{j}"
            )
            chunk = {
                "chunk_id": chunk_id,
                "pdf_source": pdf_source,
                "pasal": pasal_num,
                "bab": bab,
                "text": sub[:hard_cap],
                "length": len(block),
                "char_len": len(sub[:hard_cap]),
                "char_offset": start,          # offset blok pasal di full_text (utk routing UU)
                "is_subchunk": n_sub > 1,
                "sub_index": j,
                "n_subchunks": n_sub,
            }
            if map_klaster_by_bab:
                chunk["klaster_id"] = klaster_id
                chunk["klaster_label"] = klaster_label
                chunk["jenis"] = "batang_tubuh"   # default; di-refine oleh refine_uu_klaster
            chunks.append(chunk)

    return chunks


# ==================== UU 6/2023: routing Penjelasan -> klaster benar ====================
# Masalah: PENJELASAN besar (pasal-demi-pasal) ada SETELAH BAB XV & tak pakai header BAB,
# jadi kalau naif semua chunk Penjelasan mewarisi bab=XV -> nyasar ke klaster_15.
# Fix: (1) deteksi batas Penjelasan, (2) bangun backbone pasal-Cipta-Kerja (1..186, berurutan)
#      dari batang tubuh -> peta own_pasal->klaster, (3) route tiap chunk Penjelasan via
#      pasal yang dijelaskannya (tracking monotonic; referensi UU sektor = noise, di-skip).

def find_penjelasan_offset(full_text: str) -> int:
    """Offset awal PENJELASAN besar = 'PENJELASAN ATAS' pertama SETELAH BAB terakhir."""
    bab_bounds = find_bab_boundaries(full_text)
    last_bab_off = bab_bounds[-1][0] if bab_bounds else 0
    for m in re.finditer(r"PENJELASAN\s*\n?\s*ATAS", full_text):
        if m.start() > last_bab_off:
            return m.start()
    ms = [m.start() for m in re.finditer(r"\bPENJELASAN\b", full_text)]
    return ms[-1] if ms else len(full_text)


def build_own_pasal_backbone(full_text: str, pen_offset: int) -> list[tuple[int, int]]:
    """
    Peta breakpoint (own_pasal_start, klaster_id) per BAB di batang tubuh.

    Robust: dijangkar ke 15 header BAB (terdeteksi bersih), BUKAN rantai berurutan yang
    rapuh terhadap glitch OCR. Untuk tiap BAB (dedupe first-occurrence, buang stray duplikat),
    ambil 'Pasal N' standalone PERTAMA setelah header BAB itu sebagai own_pasal awal klaster.
    Guard monotonic (num > prev) supaya stray/embedded tidak merusak urutan.

    Return list (own_pasal_start, klaster_id) urut naik own_pasal.
    Contoh: [(1,1),(2,2),(4,3),(80,4),(85,5),...] -> klaster_for_own_pasal binary-lookup.
    """
    matches = list(PASAL_REGEX.finditer(full_text))
    # Dedupe BAB: keep first occurrence per roman (buang stray duplikat spt 'BAB V' kedua)
    seen: set[str] = set()
    ordered_bab: list[tuple[int, str]] = []
    for off, roman in find_bab_boundaries(full_text):
        if off >= pen_offset:
            break
        if roman not in seen:
            seen.add(roman)
            ordered_bab.append((off, roman))

    backbone: list[tuple[int, int]] = []
    prev = -1
    for off, roman in ordered_bab:
        kid = roman_to_int(roman)
        if kid is None:
            continue
        for m in matches:
            if m.start() > off:
                num = int(m.group(1))
                if prev < num <= 250:          # monotonic + plausible
                    backbone.append((num, kid))
                    prev = num
                break
    return backbone


def klaster_for_own_pasal(n: int, backbone: list[tuple[int, int]]) -> Optional[int]:
    """Klaster dari own_pasal terbesar di backbone yang <= n."""
    kid = None
    for own, k in backbone:
        if own <= n:
            kid = k
        else:
            break
    return kid


def refine_uu_klaster(chunks: list[dict], full_text: str) -> dict:
    """
    Refine klaster UU 6/2023 in-place:
      - tandai jenis (batang_tubuh / penjelasan) via offset batas Penjelasan
      - batang tubuh: pakai klaster dari BAB (sudah ada); bab=None -> klaster_1 (front matter)
      - penjelasan: route ke klaster benar via own-pasal yang dijelaskan (tracking monotonic)
    Return ringkasan (offset, ukuran backbone) untuk laporan verifikasi.
    """
    pen_offset = find_penjelasan_offset(full_text)
    backbone = build_own_pasal_backbone(full_text, pen_offset)

    # Batang tubuh: klaster dari BAB (sudah di-set). Penjelasan: keyword routing (per-pasal
    # tidak reliable karena didominasi referensi UU sektor setelah own Pasal ~17).
    n_penjelasan = 0
    for c in chunks:
        off = c.get("char_offset", 0)
        if off >= pen_offset:
            c["jenis"] = "penjelasan"
            k = route_penjelasan_by_keyword(c["text"], fallback_klaster=3)
            c["klaster_id"] = k
            c["klaster_label"] = UU_6_2023_KLASTER_LABELS.get(k)
            n_penjelasan += 1
        else:
            c["jenis"] = "batang_tubuh"
            if c.get("klaster_id") is None:      # bab=None front matter -> Ketentuan Umum
                c["klaster_id"] = 1
                c["klaster_label"] = UU_6_2023_KLASTER_LABELS[1]

    return {
        "penjelasan_offset": pen_offset,
        "penjelasan_offset_pct": round(100 * pen_offset / max(len(full_text), 1), 1),
        "n_penjelasan_chunks": n_penjelasan,
        "penjelasan_routing": "keyword",
        "backbone_size": len(backbone),
        "backbone_own_pasal_max": backbone[-1][0] if backbone else 0,
        "backbone_klaster_transitions": _klaster_transitions(backbone),
    }


def _klaster_transitions(backbone: list[tuple[int, int]]) -> list[dict]:
    """Ringkas backbone jadi rentang own_pasal per klaster (utk verifikasi)."""
    out: list[dict] = []
    for own, k in backbone:
        if out and out[-1]["klaster_id"] == k:
            out[-1]["own_pasal_end"] = own
        else:
            out.append({"klaster_id": k, "own_pasal_start": own, "own_pasal_end": own})
    return out

## A.5 — Bangun Chunk dari 4 PDF

Pipeline `load → clean → chunk` dijalankan untuk keempat PDF. `UU_6_2023` mendapat
pemetaan klaster (`map_klaster_by_bab=True`) lalu di-*refine* (batang tubuh per-BAB,
Penjelasan per-keyword).

In [9]:
CS = RAG_CONFIG["chunker"]

def build_chunks_for_pdf(code_name, pdf_path):
    is_uu = code_name == "UU_6_2023"
    pages = load_pdf_pages(pdf_path, strategy="pypdf")
    full = "\n".join(clean_pages(pages))
    chunks = chunk_by_pasal(
        full, code_name,
        include_bab_metadata=True,
        chunk_size=CS["chunk_size"],
        chunk_overlap=CS["chunk_overlap"],
        hard_cap=CS["hard_cap"],
        skip_cukup_jelas=True,
        map_klaster_by_bab=is_uu,
    )
    refine = refine_uu_klaster(chunks, full) if is_uu else None
    return chunks, refine

all_chunks = {}
t0 = time.time()
for name, path in PDF_FILES.items():
    ch, refine = build_chunks_for_pdf(name, path)
    all_chunks[name] = ch
    n_sub = sum(1 for c in ch if c["is_subchunk"])
    print(f"{name:12s}: {len(ch):5d} chunks (whole={len(ch)-n_sub}, sub={n_sub})")
print(f"\nTotal {sum(len(v) for v in all_chunks.values())} chunks dalam {time.time()-t0:.1f}s")

PP_5_2021   :   749 chunks (whole=440, sub=309)
PP_35_2021  :   106 chunks (whole=70, sub=36)
PP_51_2023  :    39 chunks (whole=7, sub=32)
UU_6_2023   :  2125 chunks (whole=1359, sub=766)

Total 3019 chunks dalam 35.4s


**Bukti overlap eksplisit.** Untuk pasal yang di-sub-split, potongan berurutan berbagi
tepat 100 karakter (`chunk_overlap`). Sel berikut memverifikasi invarian
`sub[k][-100:] == sub[k+1][:100]`.

In [10]:
# Verifikasi overlap eksplisit pada sub-chunk
def verify_overlap(chunks, overlap, n=3):
    from collections import defaultdict
    groups = defaultdict(list)
    for c in chunks:
        if c["n_subchunks"] > 1:
            groups[c["chunk_id"].rsplit("_sub", 1)[0]].append(c)
    shown = 0
    for key, g in groups.items():
        g = sorted(g, key=lambda x: x["sub_index"])
        for a, b in zip(g, g[1:]):
            ok = a["text"][-overlap:] == b["text"][:overlap]
            print(f"  {a['chunk_id']} -> {b['chunk_id']}: overlap {overlap} char cocok = {ok}")
            shown += 1
            if shown >= n:
                return
verify_overlap(all_chunks["PP_35_2021"], CS["chunk_overlap"], n=3)

  PP_35_2021_pasal_1_0_sub0 -> PP_35_2021_pasal_1_0_sub1: overlap 100 char cocok = True
  PP_35_2021_pasal_1_0_sub1 -> PP_35_2021_pasal_1_0_sub2: overlap 100 char cocok = True
  PP_35_2021_pasal_1_0_sub2 -> PP_35_2021_pasal_1_0_sub3: overlap 100 char cocok = True


**Distribusi klaster UU 6/2023.** Ke-18 koleksi (3 PP + 15 klaster UU) harus terisi.
Batang tubuh dipetakan presisi per-BAB; Penjelasan di-route per-keyword.

In [11]:
# Ringkasan chunk UU per klaster + jenis
from collections import Counter
uu = all_chunks["UU_6_2023"]
kd = Counter(c.get("klaster_id") for c in uu)
jd = Counter(c.get("jenis") for c in uu)
print("Jenis:", dict(jd))
print("Char per chunk max:", max(c["char_len"] for c in uu), "(harus <= chunk_size)")
print("\nDistribusi klaster UU_6_2023:")
for k in range(1, 16):
    label = UU_6_2023_KLASTER_LABELS[k]
    print(f"  klaster {k:2d} ({label[:34]:34s}): {kd.get(k, 0)}")
assert all(kd.get(k, 0) > 0 for k in range(1, 16)), "Ada klaster kosong!"
print("\nSemua 15 klaster UU terisi.")

Jenis: {'batang_tubuh': 1442, 'penjelasan': 683}
Char per chunk max: 1200 (harus <= chunk_size)

Distribusi klaster UU_6_2023:
  klaster  1 (Ketentuan Umum                    ): 22
  klaster  2 (Asas, Tujuan, dan Ruang Lingkup   ): 4
  klaster  3 (Peningkatan Ekosistem Investasi da): 1625
  klaster  4 (Ketenagakerjaan                   ): 105
  klaster  5 (Koperasi dan UMK-M                ): 55
  klaster  6 (Kemudahan Berusaha                ): 134
  klaster  7 (Dukungan Riset dan Inovasi        ): 2
  klaster  8 (Pengadaan Tanah                   ): 51
  klaster  9 (Kawasan Ekonomi                   ): 47
  klaster 10 (Investasi Pemerintah Pusat dan Pro): 31
  klaster 11 (Administrasi Pemerintahan         ): 25
  klaster 12 (Pengawasan dan Pembinaan          ): 4
  klaster 13 (Ketentuan Lain-Lain               ): 2
  klaster 14 (Ketentuan Peralihan               ): 3
  klaster 15 (Ketentuan Penutup                 ): 15

Semua 15 klaster UU terisi.


## A.6 — Embedder BGE-M3

`BAAI/bge-m3` adalah model embedding *open-source* multilingual (mendukung Bahasa
Indonesia native, dimensi 1024, konteks hingga 8k token). Model di-*load sekali* dan
di-cache di memori — tidak di-reload per query.

Model diunduh via **`aria2c`** (16 koneksi paralel + resume) karena downloader HF bawaan
kadang *hang* pada file bobot besar (LFS) di sebagian environment Colab.

In [12]:
# Helper unduh model + CACHE PERMANEN DI DRIVE.
# Strategi: download SEKALI via aria2c ke /content (write cepat), lalu SALIN ke Drive.
# Sesi berikutnya (walau VM/akun ganti) -> load dari Drive instan, TIDAK download lagi.
# aria2c "gigih" (resume + banyak retry + bunuh koneksi stall + header token utk LFS);
# fallback hf_hub_download hanya last-resort.
import subprocess, time
from huggingface_hub import list_repo_files, get_token, hf_hub_download

MODELS_LOCAL = "/content/models"
MODELS_DRIVE = str(BASE / "models")   # cache permanen di Google Drive

def _model_complete(d):
    if not os.path.isdir(d):
        return False
    files = os.listdir(d)
    return ("config.json" in files) and any(f.endswith((".safetensors", ".bin")) for f in files)

def _aria2_fetch(url, out_dir, name, token, outer_tries=3):
    cmd = ["aria2c", "-c", "-x", "16", "-s", "16",
           "--allow-overwrite=true", "--auto-file-renaming=false",
           "--max-tries=50", "--retry-wait=5", "--timeout=60", "--connect-timeout=60",
           "--lowest-speed-limit=100K",   # koneksi < 100KB/s dianggap stall -> retry koneksi baru
           "--console-log-level=warn", "--summary-interval=2",
           "-d", out_dir, "-o", name, url]
    if token:
        cmd += ["--header", f"Authorization: Bearer {token}"]
    for i in range(outer_tries):
        if subprocess.run(cmd).returncode == 0:
            return True
        print(f"    aria2c retry {i + 1}/{outer_tries} untuk {name}...")
        time.sleep(5)
    return False

def ensure_model(repo, name, skip=()):
    """Return path lokal model. Download sekali -> cache di Drive -> sesi berikutnya instan."""
    local_dir = os.path.join(MODELS_LOCAL, name)
    drive_dir = os.path.join(MODELS_DRIVE, name)
    os.makedirs(local_dir, exist_ok=True)
    if _model_complete(local_dir):
        print(f"  [lokal] {repo} sudah ada"); return local_dir
    if _model_complete(drive_dir):
        print(f"  [cache Drive] {repo} -> salin Drive ke lokal ...")
        subprocess.run(f"cp -rn '{drive_dir}/.' '{local_dir}/'", shell=True)
        return local_dir
    # belum ada di mana pun -> download via aria2c ke lokal
    token = get_token()   # WAJIB utk file LFS (dari login HF / Secret HF_TOKEN)
    base = f"https://huggingface.co/{repo}/resolve/main"
    for f in list_repo_files(repo):
        if any(s in f for s in skip):
            continue
        out = os.path.join(local_dir, os.path.dirname(f)); os.makedirs(out, exist_ok=True)
        if not _aria2_fetch(f"{base}/{f}", out, os.path.basename(f), token):
            print(f"  aria2c gagal utk {f} -> fallback hf_hub_download")
            hf_hub_download(repo, f, local_dir=local_dir)
    # salin ke Drive utk sesi berikutnya (write sekuensial -> aman di FUSE)
    os.makedirs(drive_dir, exist_ok=True)
    subprocess.run(f"cp -rn '{local_dir}/.' '{drive_dir}/'", shell=True)
    print(f"  [cache Drive] {repo} disimpan ke Drive (sesi berikutnya instan)")
    return local_dir
print("Helper ensure_model (aria2c + cache Drive) siap.")

Helper ensure_model (aria2c + cache Drive) siap.


In [13]:
from sentence_transformers import SentenceTransformer

EMB = RAG_CONFIG["embedder"]
_bge_dir = ensure_model(
    EMB["model_id"], "bge-m3",
    skip=("onnx", "colbert_linear", "sparse_linear", "imgs/", ".gitattributes", "README"))
embedder = SentenceTransformer(_bge_dir, device="cuda" if torch.cuda.is_available() else "cpu")

def embed_texts(texts):
    return embedder.encode(
        texts, batch_size=EMB["batch_size"],
        normalize_embeddings=EMB["normalize"],
        convert_to_numpy=True, show_progress_bar=False,
    )

# sanity: dimensi vektor
_dim = embed_texts(["uji dimensi embedding"]).shape[1]
print(f"Embedder {EMB['model_id']} siap (via aria2c). Dimensi vektor: {_dim}")

  [cache Drive] BAAI/bge-m3 -> salin Drive ke lokal ...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedder BAAI/bge-m3 siap (via aria2c). Dimensi vektor: 1024


## A.7 — Vector Store: ChromaDB (18 koleksi)

Strategi **multi-koleksi**: 3 koleksi untuk PP + **15 koleksi per klaster** untuk
`UU_6_2023`. Pemisahan per-klaster membuat retrieval ter-fokus dan mendukung *metadata
filtering* pada tier berikutnya. Indeks disimpan **persisten** ke Drive.

In [14]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Nama 18 koleksi
COLLECTION_NAMES = ["pp_5_2021", "pp_35_2021", "pp_51_2023"] + \
                   [f"uu_6_2023_klaster_{k}" for k in range(1, 16)]

def collection_for(chunk):
    src = chunk["pdf_source"]
    if src == "UU_6_2023":
        return f"uu_6_2023_klaster_{chunk['klaster_id']}"
    return src.lower()   # PP_5_2021 -> pp_5_2021

def chunk_metadata(chunk):
    # ChromaDB metadata hanya menerima str/int/float/bool (tanpa None)
    return {
        "pdf_source": chunk["pdf_source"],
        "pasal": str(chunk["pasal"]),
        "bab": chunk.get("bab") or "",
        "klaster_id": int(chunk["klaster_id"]) if chunk.get("klaster_id") else 0,
        "klaster_label": chunk.get("klaster_label") or "",
        "jenis": chunk.get("jenis") or "batang_tubuh",
        "is_subchunk": bool(chunk["is_subchunk"]),
    }

# Reset koleksi lama (idempoten saat Run All ulang) lalu buat 18 koleksi kosong
for name in COLLECTION_NAMES:
    try:
        client.delete_collection(name)
    except Exception:
        pass
collections = {
    name: client.create_collection(name, metadata={"hnsw:space": RAG_CONFIG["vector_store"]["distance_metric"]})
    for name in COLLECTION_NAMES
}
print(f"{len(collections)} koleksi ChromaDB dibuat di {CHROMA_DIR}")

18 koleksi ChromaDB dibuat di /content/drive/MyDrive/PGABL/chroma_db


**Ingest.** Setiap chunk di-*embed* lalu dimasukkan ke koleksi tujuannya beserta
metadata (pdf, pasal, bab, klaster, jenis).

In [15]:
# Kelompokkan chunk per koleksi, lalu embed & masukkan per batch
from collections import defaultdict
buckets = defaultdict(list)
for name, chunks in all_chunks.items():
    for c in chunks:
        buckets[collection_for(c)].append(c)

t0 = time.time()
for cname, chunks in buckets.items():
    texts = [c["text"] for c in chunks]
    embs = embed_texts(texts)
    collections[cname].add(
        ids=[c["chunk_id"] for c in chunks],
        embeddings=[e.tolist() for e in embs],
        documents=texts,
        metadatas=[chunk_metadata(c) for c in chunks],
    )
    print(f"  {cname:24s}: {len(chunks)} chunk")
print(f"\nIngest selesai dalam {time.time()-t0:.1f}s")

  pp_5_2021               : 749 chunk
  pp_35_2021              : 106 chunk
  pp_51_2023              : 39 chunk
  uu_6_2023_klaster_1     : 22 chunk
  uu_6_2023_klaster_2     : 4 chunk
  uu_6_2023_klaster_3     : 1625 chunk
  uu_6_2023_klaster_4     : 105 chunk
  uu_6_2023_klaster_5     : 55 chunk
  uu_6_2023_klaster_6     : 134 chunk
  uu_6_2023_klaster_7     : 2 chunk
  uu_6_2023_klaster_8     : 51 chunk
  uu_6_2023_klaster_9     : 47 chunk
  uu_6_2023_klaster_10    : 31 chunk
  uu_6_2023_klaster_11    : 25 chunk
  uu_6_2023_klaster_12    : 4 chunk
  uu_6_2023_klaster_13    : 2 chunk
  uu_6_2023_klaster_14    : 3 chunk
  uu_6_2023_klaster_15    : 15 chunk

Ingest selesai dalam 115.7s


In [16]:
# Verifikasi: 18 koleksi, semua count > 0
listed = client.list_collections()
print(f"Jumlah koleksi: {len(listed)} (target 18)")
total = 0
for name in COLLECTION_NAMES:
    cnt = client.get_collection(name).count()
    total += cnt
    print(f"  {name:24s}: {cnt}")
print(f"\nTotal vektor terindeks: {total}")
assert len(listed) == 18 and all(client.get_collection(n).count() > 0 for n in COLLECTION_NAMES)
print("18 koleksi terisi.")

Jumlah koleksi: 18 (target 18)
  pp_5_2021               : 749
  pp_35_2021              : 106
  pp_51_2023              : 39
  uu_6_2023_klaster_1     : 22
  uu_6_2023_klaster_2     : 4
  uu_6_2023_klaster_3     : 1625
  uu_6_2023_klaster_4     : 105
  uu_6_2023_klaster_5     : 55
  uu_6_2023_klaster_6     : 134
  uu_6_2023_klaster_7     : 2
  uu_6_2023_klaster_8     : 51
  uu_6_2023_klaster_9     : 47
  uu_6_2023_klaster_10    : 31
  uu_6_2023_klaster_11    : 25
  uu_6_2023_klaster_12    : 4
  uu_6_2023_klaster_13    : 2
  uu_6_2023_klaster_14    : 3
  uu_6_2023_klaster_15    : 15

Total vektor terindeks: 3019
18 koleksi terisi.


## A.8 — Generator: Llama-3.2-3B fine-tuned

Generator adalah model hasil fine-tuning K1 (`PGABL-Llama-3.2-3B-GRPO`, *merged 16-bit*,
publik di Hugging Face). Dimuat dalam **4-bit** (QLoRA-style NF4) agar hemat VRAM dan muat
bersama BGE-M3 di T4. Model di-*load sekali*.

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

GEN = RAG_CONFIG["generator"]
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
_gen_dir = ensure_model(
    GEN["model_id"], "grpo",
    skip=(".gitattributes", "README", "original/", ".onnx", ".pth"))
gen_tokenizer = AutoTokenizer.from_pretrained(_gen_dir)
gen_model = AutoModelForCausalLM.from_pretrained(
    _gen_dir, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16,
)
gen_model.eval()
print(f"Generator {GEN['model_id']} dimuat (4-bit, via aria2c).")

  [cache Drive] nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO disimpan ke Drive (sesi berikutnya instan)


[transformers] The tokenizer you are loading from '/content/models/grpo' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Generator nazhifsetya-merpati/PGABL-Llama-3.2-3B-GRPO dimuat (4-bit, via aria2c).


In [18]:
@torch.no_grad()
def llm_generate(system_prompt, user_prompt, max_new_tokens=None, fewshot=None, temperature=None):
    # fewshot: list [(contoh_user, contoh_assistant), ...] untuk guided decoding
    messages = [{"role": "system", "content": system_prompt}]
    for ex_user, ex_assistant in (fewshot or []):
        messages.append({"role": "user", "content": ex_user})
        messages.append({"role": "assistant", "content": ex_assistant})
    messages.append({"role": "user", "content": user_prompt})
    # apply_chat_template bisa mengembalikan tensor ATAU BatchEncoding (tergantung versi
    # transformers). Tangani keduanya secara robust.
    enc = gen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    if isinstance(enc, dict) or hasattr(enc, "input_ids"):
        input_ids = enc["input_ids"].to(gen_model.device)
        gen_kwargs = {"input_ids": input_ids}
        if "attention_mask" in enc:
            gen_kwargs["attention_mask"] = enc["attention_mask"].to(gen_model.device)
    else:
        input_ids = enc.to(gen_model.device)
        gen_kwargs = {"input_ids": input_ids}
    input_len = input_ids.shape[-1]
    out = gen_model.generate(
        **gen_kwargs,
        max_new_tokens=max_new_tokens or GEN["max_new_tokens"],
        do_sample=True,
        temperature=temperature if temperature is not None else GEN["temperature"],
        top_p=GEN["top_p"], repetition_penalty=GEN["repetition_penalty"],
        pad_token_id=gen_tokenizer.eos_token_id,
    )
    return gen_tokenizer.decode(
        out[0][input_len:], skip_special_tokens=True,
        clean_up_tokenization_spaces=False,   # BPE: cleanup destruktif, matikan
    ).strip()

def strip_think(text):
    # Model fine-tuned dapat menyertakan blok <think>...</think>; ambil jawaban final saja
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

## A.9 — Pipeline RAG Dasar

Alur tier **Basic**: *embed* pertanyaan → cari Top-K chunk termirip di seluruh koleksi →
susun konteks → *generate* jawaban dengan model fine-tuned.

In [19]:
def retrieve(query, k=None, collection_names=None):
    k = k or RAG_CONFIG["retriever"]["top_k"]
    q_emb = embed_texts([query])[0].tolist()
    names = collection_names or COLLECTION_NAMES
    candidates = []
    for name in names:
        col = client.get_collection(name)
        n = col.count()
        if n == 0:
            continue
        res = col.query(query_embeddings=[q_emb], n_results=min(k, n))
        for i in range(len(res["ids"][0])):
            candidates.append({
                "chunk_id": res["ids"][0][i],
                "document": res["documents"][0][i],
                "metadata": res["metadatas"][0][i],
                "distance": res["distances"][0][i],
                "collection": name,
            })
    candidates.sort(key=lambda x: x["distance"])
    return candidates[:k]

def format_sources(hits):
    return "\n\n".join(f"[{i+1}] {h['document']}" for i, h in enumerate(hits))

def rag_basic(query, k=None):
    hits = retrieve(query, k=k)
    user_prompt = RAG_CONFIG["context_template"].format(
        sources=format_sources(hits), question=query)
    raw = llm_generate(RAG_CONFIG["system_prompt"], user_prompt)
    return {"query": query, "answer": strip_think(raw), "raw": raw, "sources": hits}

### Uji coba 3 pertanyaan

Termasuk pertanyaan uji wajib **hak lembur staf admin**. Untuk tiap query ditampilkan
sumber yang di-retrieve (pdf, pasal, klaster, jarak) dan jawaban model.

In [20]:
TEST_QUERIES = [
    "Berapa upah kerja lembur untuk pekerja pada jam pertama?",
    "Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?",
    "Bagaimana formula penyesuaian upah minimum menurut regulasi terbaru?",
]

for q in TEST_QUERIES:
    out = rag_basic(q)
    print("=" * 90)
    print("TANYA :", q)
    print("-" * 90)
    for i, h in enumerate(out["sources"]):
        m = h["metadata"]
        print(f"  [{i+1}] {m['pdf_source']} Pasal {m['pasal']} "
              f"| {m['klaster_label'] or m['bab'] or '-'} | jarak={h['distance']:.3f}")
    print("-" * 90)
    print("JAWAB :", out["answer"])
    print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


TANYA : Berapa upah kerja lembur untuk pekerja pada jam pertama?
------------------------------------------------------------------------------------------
  [1] PP_35_2021 Pasal 31 | IV | jarak=0.272
  [2] PP_35_2021 Pasal 31 | IV | jarak=0.285
  [3] PP_35_2021 Pasal 1 | I | jarak=0.312
  [4] PP_35_2021 Pasal 32 | IV | jarak=0.317
  [5] UU_6_2023 Pasal 78 | Ketenagakerjaan | jarak=0.340
------------------------------------------------------------------------------------------
JAWAB : Menurut Pasal 31 Ayat (1), Upah Kerja Lembur untuk pekerja pada jam pertama adalah 1,5 kali Upah sejam.



[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?
------------------------------------------------------------------------------------------
  [1] PP_5_2021 Pasal 176 | IV | jarak=0.339
  [2] PP_5_2021 Pasal 12 | II | jarak=0.340
  [3] PP_5_2021 Pasal 177 | IV | jarak=0.363
  [4] PP_5_2021 Pasal 15 | II | jarak=0.372
  [5] PP_5_2021 Pasal 14 | II | jarak=0.388
------------------------------------------------------------------------------------------
JAWAB : Nomor Induk Berusaha (NIB) adalah identitas unik yang diterbitkan oleh Lembaga OSS untuk setiap Pelaku Usaha. Ini mencakup data seperti profil, permodalan usaha, nomor pokok wajib pajak, KBLI, dan lokasi usaha. NIB digunakan sebagai bukti registrasi/pendaftaran Pelaku Usaha untuk melakukan kegiatan usaha dan berfungsi sebagai identitas bagi Pelaku Usaha. Selain itu, NIB juga berfungsi sebagai angka pengenal impor, hak akses kepabeanan, pendaftaran kepesertaan Pelaku Usaha untuk jaminan sosial kesehatan dan jaminan sosial 

---

**Ringkasan Bagian A (RAG Dasar / K2 Basic).** Pipeline end-to-end telah berjalan:
4 PDF → chunk per-pasal dengan overlap eksplisit → embedding BGE-M3 → 18 koleksi ChromaDB
persisten → retrieval Top-K → generasi jawaban dengan Llama-3.2-3B fine-tuned.

# Bagian B — RAG Skilled

Meningkatkan retrieval dengan empat komponen:

1. **Query routing + metadata filtering** — kata kunci pertanyaan menentukan klaster/koleksi
   kandidat (mis. *lembur* → Ketenagakerjaan). Bila tak ada kecocokan, retrieval jatuh ke
   seluruh 18 koleksi (*fallback-to-all*) agar recall terjaga.
2. **Ensemble Retriever (BM25 + Dense)** — BM25 menangkap kecocokan kata eksak (nomor pasal,
   istilah "PKWT"); *dense* (BGE-M3) menangkap kemiripan makna. Keduanya digabung dengan
   **Reciprocal Rank Fusion (RRF)**.
3. **Parent-Child Retriever** — *child* (potongan presisi) dipakai untuk mencari; konteks
   yang diberikan ke generator adalah **pasal utuh** (*parent*, hasil rekonstruksi de-overlap
   dari sub-chunk).
4. **Sitasi sumber** — setiap konteks dilabeli `[Sumber N: PDF - Klaster - Pasal]`, dan model
   diinstruksikan mengutip `[Sumber N]` pada jawabannya.

## B.1 — Layer Retriever (BM25, router, RRF, parent-store)

Modul retriever di-*inline* agar notebook mandiri.

In [21]:
# --- Retriever (BM25 + query routing + RRF + parent-child + sitasi) ---
"""
PGABL - Retriever layer (Tahap 3b Skilled).

Komponen (semua modular, portable ke domain lain):
  - tokenize + BM25Index    : sinyal sparse (exact-match: nomor pasal, istilah "PKWT")
  - route_query             : keyword -> klaster/collection kandidat (metadata filtering)
  - reciprocal_rank_fusion  : gabung ranking BM25 + Dense (RRF)
  - build_parent_store      : rekonstruksi teks pasal UTUH (parent) dari sub-chunk (child)
  - format_citation         : label sitasi [Sumber N: PDF - BAB/Klaster - Pasal]

Dense retrieval (BGE-M3 + ChromaDB) dijalankan di notebook (butuh GPU); modul ini
menyediakan logika non-model yang di-verify lokal via scripts/10_verify_skilled.py.
"""

from __future__ import annotations
import re
from collections import defaultdict
from typing import Optional


# ==================== Tokenizer untuk BM25 ====================
_TOKEN_RE = re.compile(r"[a-z0-9]+")


def tokenize(text: str) -> list[str]:
    """Tokenisasi sederhana: lowercase + ambil token alfanumerik."""
    return _TOKEN_RE.findall(text.lower())


# ==================== Query routing (metadata filtering) ====================
# Keyword tema -> collection kandidat. PP dipetakan by topik:
#   PP_5_2021 = perizinan, PP_35_2021 = ketenagakerjaan, PP_51_2023 = pengupahan.
ROUTING_RULES = [
    {
        "theme": "ketenagakerjaan",
        "keywords": ["lembur", "upah", "pkwt", "pkwtt", "phk", "pemutusan hubungan kerja",
                     "pesangon", "cuti", "pekerja", "buruh", "serikat", "alih daya",
                     "outsourcing", "waktu kerja", "pengupahan", "tenaga kerja",
                     "jaminan sosial", "perjanjian kerja", "hubungan kerja"],
        "collections": ["pp_35_2021", "pp_51_2023", "uu_6_2023_klaster_4"],
    },
    {
        "theme": "perizinan_investasi",
        "keywords": ["izin", "perizinan", "nib", "oss", "kbli", "risiko", "berusaha",
                     "penanaman modal", "investasi", "lingkungan hidup", "bangunan gedung",
                     "tata ruang", "amdal"],
        "collections": ["pp_5_2021", "uu_6_2023_klaster_3", "uu_6_2023_klaster_6"],
    },
    {
        "theme": "pengupahan",
        "keywords": ["upah minimum", "ump", "umk", "umr", "formula upah",
                     "penyesuaian upah", "dewan pengupahan"],
        "collections": ["pp_51_2023", "uu_6_2023_klaster_4"],
    },
]


def route_query(query: str, all_collections: list[str]) -> tuple[list[str], Optional[str]]:
    """
    Return (collections_target, theme). Kalau ada keyword match -> collection kandidat
    tema tsb (metadata filter). Kalau tidak ada match -> semua collection (fallback-to-all).
    """
    low = query.lower()
    matched: list[str] = []
    theme = None
    for rule in ROUTING_RULES:
        if any(kw in low for kw in rule["keywords"]):
            theme = theme or rule["theme"]
            for c in rule["collections"]:
                if c in all_collections and c not in matched:
                    matched.append(c)
    if not matched:
        return list(all_collections), None
    return matched, theme


# ==================== Reciprocal Rank Fusion ====================
def reciprocal_rank_fusion(ranked_lists: list[list[str]], k: int = 60) -> list[tuple[str, float]]:
    """
    Gabung beberapa ranked list (list of ids urut relevansi) via RRF.
    Skor(id) = sum_over_lists 1 / (k + rank), rank 0-based.
    Return list (id, score) urut skor menurun.
    """
    scores: dict[str, float] = defaultdict(float)
    for lst in ranked_lists:
        for rank, item_id in enumerate(lst):
            scores[item_id] += 1.0 / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# ==================== Parent-Child ====================
def parent_key_of(chunk_id: str) -> str:
    """child chunk_id -> parent_key (buang suffix _sub{j})."""
    return chunk_id.rsplit("_sub", 1)[0]


def build_parent_store(all_chunks: dict[str, list[dict]], overlap: int = 100) -> dict[str, dict]:
    """
    Bangun 'parent' = teks pasal UTUH dari sub-chunk (child). Rekonstruksi de-overlap:
    parent = sub0 + sub1[overlap:] + sub2[overlap:] + ...  (kebalikan sliding-window split,
    jadi parent == blok pasal asli). Untuk pasal 1-chunk, parent == child.

    Return {parent_key: {parent_id, text, pdf_source, pasal, bab, klaster_label, n_children}}.
    """
    groups: dict[str, list[dict]] = defaultdict(list)
    for chunks in all_chunks.values():
        for c in chunks:
            groups[parent_key_of(c["chunk_id"])].append(c)

    store: dict[str, dict] = {}
    for pkey, g in groups.items():
        g = sorted(g, key=lambda x: x["sub_index"])
        text = g[0]["text"]
        for c in g[1:]:
            text += c["text"][overlap:] if len(c["text"]) > overlap else ""
        first = g[0]
        store[pkey] = {
            "parent_id": pkey,
            "text": text,
            "pdf_source": first["pdf_source"],
            "pasal": first["pasal"],
            "bab": first.get("bab"),
            "klaster_label": first.get("klaster_label"),
            "n_children": len(g),
        }
    return store


# ==================== BM25 index ====================
class BM25Index:
    """BM25Okapi di atas seluruh chunk (global). Filter collection via allowed_ids."""

    def __init__(self, chunks: list[dict]):
        from rank_bm25 import BM25Okapi
        self.chunks = chunks
        self.ids = [c["chunk_id"] for c in chunks]
        self.bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

    def search(self, query: str, top_n: int = 20, allowed_ids: Optional[set] = None) -> list[str]:
        scores = self.bm25.get_scores(tokenize(query))
        order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        out: list[str] = []
        for i in order:
            cid = self.ids[i]
            if allowed_ids is not None and cid not in allowed_ids:
                continue
            if scores[i] <= 0:
                break
            out.append(cid)
            if len(out) >= top_n:
                break
        return out


# ==================== Citation formatting ====================
def format_citation(rank: int, meta: dict) -> str:
    """Label sitasi utk 1 sumber: [Sumber N: PDF - BAB/Klaster - Pasal X]."""
    pdf = meta.get("pdf_source", "?")
    scope = meta.get("klaster_label") or (f"BAB {meta['bab']}" if meta.get("bab") else "")
    pasal = meta.get("pasal", "?")
    scope_part = f" - {scope}" if scope else ""
    return f"[Sumber {rank}: {pdf}{scope_part} - Pasal {pasal}]"

In [22]:
from collections import defaultdict

# Index sparse (BM25) global + parent store + peta koleksi -> chunk_ids
_flat_chunks = [c for chunks in all_chunks.values() for c in chunks]
bm25 = BM25Index(_flat_chunks)
parent_store = build_parent_store(all_chunks, overlap=RAG_CONFIG["chunker"]["chunk_overlap"])

collection_ids = defaultdict(set)
for chunks in all_chunks.values():
    for c in chunks:
        collection_ids[collection_for(c)].add(c["chunk_id"])

print(f"BM25 index : {len(_flat_chunks)} chunk")
print(f"Parent store: {len(parent_store)} pasal (parent) — "
      f"{sum(1 for p in parent_store.values() if p['n_children']>1)} pasal ter-split")

BM25 index : 3019 chunk
Parent store: 2295 pasal (parent) — 419 pasal ter-split


## B.2 — Ensemble Retrieve + Parent-Child

`ensemble_retrieve` menjalankan dense (Chroma, terfilter koleksi hasil routing) dan BM25
(terfilter ke id koleksi yang sama), lalu menggabung dengan RRF. `retrieve_with_parents`
mengubah *child* hasil fusi menjadi *parent* (pasal utuh) yang unik untuk konteks generator.

In [23]:
def dense_search(query, collections, top_n=20):
    q_emb = embed_texts([query])[0].tolist()
    cand = []
    for name in collections:
        col = client.get_collection(name)
        n = col.count()
        if n == 0:
            continue
        res = col.query(query_embeddings=[q_emb], n_results=min(top_n, n))
        for i in range(len(res["ids"][0])):
            cand.append((res["ids"][0][i], res["distances"][0][i]))
    cand.sort(key=lambda x: x[1])
    return [cid for cid, _ in cand[:top_n]]

def ensemble_retrieve(query, top_n=20):
    cols, theme = route_query(query, COLLECTION_NAMES)   # metadata filtering
    allowed_ids = set().union(*(collection_ids[c] for c in cols)) if cols else set()
    dense_ranked = dense_search(query, cols, top_n=top_n)
    sparse_ranked = bm25.search(query, top_n=top_n, allowed_ids=allowed_ids)
    fused = reciprocal_rank_fusion([dense_ranked, sparse_ranked], k=60)
    return {"theme": theme, "collections": cols,
            "dense": dense_ranked, "sparse": sparse_ranked,
            "fused": [cid for cid, _ in fused]}

def retrieve_with_parents(query, top_k=5, top_n=20):
    ens = ensemble_retrieve(query, top_n=top_n)
    seen, parents = set(), []
    for cid in ens["fused"]:
        pkey = parent_key_of(cid)
        if pkey in seen:
            continue
        seen.add(pkey)
        p = dict(parent_store[pkey]); p["child_id"] = cid
        parents.append(p)
        if len(parents) >= top_k:
            break
    ens["parents"] = parents
    return ens

## B.3 — Pipeline RAG Skilled (dengan sitasi)

In [24]:
import re as _re

SYSTEM_PROMPT_SKILLED = (
    "Anda adalah asisten hukum untuk tim legal internal. Jawab pertanyaan berdasarkan konteks "
    "pasal yang diberikan. Terjemahkan istilah awam pada pertanyaan ke istilah regulasi (mis. "
    "'karyawan/staf' = 'Pekerja/Buruh'). Sebutkan nomor pasal yang menjadi dasar jawaban. "
    "Gunakan Bahasa Indonesia hukum yang formal dan ringkas. Hanya jika konteks benar-benar "
    "tidak memuat jawaban, katakan: 'Informasi tidak ditemukan pada regulasi yang tersedia.'"
)
CONTEXT_TEMPLATE_SKILLED = (
    "Konteks pasal (dengan sumber):\n{sources}\n\n"
    "Pertanyaan: {question}\nJawaban:"
)

# Few-shot 2-contoh: disambiguasi ayat/kondisi + generalisasi istilah awam + sebut pasal.
FEWSHOT_SKILLED = [
    (
        "Konteks pasal (dengan sumber):\n"
        "[Sumber 1: PP_35_2021 - Ketenagakerjaan - Pasal 31]\n"
        "(1) Upah Kerja Lembur pada hari kerja: a. jam pertama sebesar 1,5 kali Upah sejam; "
        "b. jam berikutnya sebesar 2 kali Upah sejam. (2) Bila lembur pada hari istirahat "
        "mingguan/libur resmi, jam pertama sampai ketujuh dibayar 2 kali Upah sejam.\n\n"
        "Pertanyaan: Berapa upah lembur jam pertama bagi karyawan pada hari kerja biasa?\n"
        "Jawaban:",
        "Berdasarkan Pasal 31, pada hari kerja biasa upah kerja lembur untuk jam pertama "
        "adalah 1,5 kali Upah sejam."
    ),
    (
        "Konteks pasal (dengan sumber):\n"
        "[Sumber 1: PP_5_2021 - Ketentuan Umum - Pasal 1]\n"
        "Nomor Induk Berusaha yang selanjutnya disingkat NIB adalah bukti registrasi Pelaku "
        "Usaha untuk melakukan kegiatan usaha.\n\n"
        "Pertanyaan: Apa itu NIB?\n"
        "Jawaban:",
        "Berdasarkan Pasal 1, NIB (Nomor Induk Berusaha) adalah bukti registrasi Pelaku Usaha "
        "untuk melakukan kegiatan usaha."
    ),
]

def build_cited_context(parents):
    return "\n\n".join(f"{format_citation(i+1, p)}\n{p['text']}" for i, p in enumerate(parents))

def attach_citations(answer, parents):
    # Grounding deterministik: nomor pasal yang disebut model -> [Sumber N] yang cocok.
    cited, seen = [], set()
    for i, p in enumerate(parents):
        if _re.search(rf"[Pp]asal\s+0*{_re.escape(str(p['pasal']))}\b", answer) and i not in seen:
            cited.append((i + 1, p)); seen.add(i)
    if not cited and parents:            # fallback: sumber utama (Top-1)
        cited = [(1, parents[0])]
    footer = "; ".join(f"[Sumber {n}: {p['pdf_source']} Pasal {p['pasal']}]" for n, p in cited)
    return answer.rstrip() + "\n\nRujukan: " + footer

def rag_skilled(query, top_k=None, top_n=20):
    top_k = top_k or RAG_CONFIG["retriever"]["top_k"]
    r = retrieve_with_parents(query, top_k=top_k, top_n=top_n)
    user_prompt = CONTEXT_TEMPLATE_SKILLED.format(
        sources=build_cited_context(r["parents"]), question=query)
    raw = llm_generate(SYSTEM_PROMPT_SKILLED, user_prompt, fewshot=FEWSHOT_SKILLED)
    answer = attach_citations(strip_think(raw), r["parents"])
    return {"query": query, "answer": answer, "raw": raw,
            "parents": r["parents"], "theme": r["theme"],
            "collections": r["collections"], "dense": r["dense"], "sparse": r["sparse"]}

### Bukti Ensemble + Parent-Child

Sel berikut menampilkan perbedaan ranking Dense vs BM25 vs hasil fusi RRF, serta contoh
ekspansi *child* → *parent* (pasal panjang yang ter-split).

In [25]:
r = retrieve_with_parents("upah kerja lembur untuk jam pertama", top_k=5, top_n=15)
print("Routing theme :", r["theme"], "| koleksi:", r["collections"])
print("Dense  top5 :", r["dense"][:5])
print("BM25   top5 :", r["sparse"][:5])
print("RRF    top5 :", [p["child_id"] for p in r["parents"]])
print()
for p in r["parents"]:
    if p["n_children"] > 1:
        child = next(c for c in _flat_chunks if c["chunk_id"] == p["child_id"])
        print(f"Parent-Child: child {p['child_id']} ({len(child['text'])} char) "
              f"-> parent {p['parent_id']} ({len(p['text'])} char, {p['n_children']} sub-chunk)")
        break
else:
    print("(Top-5 kali ini pasal pendek — child = parent)")

Routing theme : ketenagakerjaan | koleksi: ['pp_35_2021', 'pp_51_2023', 'uu_6_2023_klaster_4']
Dense  top5 : ['PP_35_2021_pasal_31_30_sub0', 'PP_35_2021_pasal_31_30_sub1', 'PP_35_2021_pasal_32_31', 'PP_35_2021_pasal_1_0_sub1', 'PP_35_2021_pasal_26_26']
BM25   top5 : ['PP_35_2021_pasal_31_30_sub0', 'PP_35_2021_pasal_31_30_sub1', 'UU_6_2023_pasal_78_819', 'PP_35_2021_pasal_34_34', 'PP_35_2021_pasal_26_26']
RRF    top5 : ['PP_35_2021_pasal_31_30_sub0', 'UU_6_2023_pasal_78_819', 'PP_35_2021_pasal_32_31', 'PP_35_2021_pasal_26_26', 'PP_35_2021_pasal_1_0_sub1']

Parent-Child: child PP_35_2021_pasal_31_30_sub0 (1200 char) -> parent PP_35_2021_pasal_31_30 (1853 char, 2 sub-chunk)


### Uji coba dengan sitasi

Pertanyaan dari domain legal; jawaban wajib memuat rujukan `[Sumber N]`.

In [26]:
import re as _re

SKILLED_TESTS = [
    "Berapa upah kerja lembur untuk 1 jam pertama bagi staf admin?",
    "Berapa besar uang kompensasi bagi pekerja dengan PKWT?",
    "Apa yang dimaksud Perizinan Berusaha Berbasis Risiko?",
]
for q in SKILLED_TESTS:
    out = rag_skilled(q)
    print("=" * 90)
    print("TANYA :", q)
    print(f"  routing = {out['theme'] or 'ALL'} | {len(out['collections'])} koleksi")
    print("-" * 90)
    for i, p in enumerate(out["parents"]):
        print(f"  [Sumber {i+1}] {p['pdf_source']} Pasal {p['pasal']} "
              f"| {p['klaster_label'] or (('BAB ' + p['bab']) if p['bab'] else '-')}")
    print("-" * 90)
    print("JAWAB :", out["answer"])
    print("Memuat sitasi [Sumber N]:", bool(_re.search(r"\[Sumber\s*\d+", out["answer"])))
    print()

[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Berapa upah kerja lembur untuk 1 jam pertama bagi staf admin?
  routing = ketenagakerjaan | 3 koleksi
------------------------------------------------------------------------------------------
  [Sumber 1] PP_35_2021 Pasal 31 | BAB IV
  [Sumber 2] UU_6_2023 Pasal 78 | Ketenagakerjaan
  [Sumber 3] PP_35_2021 Pasal 1 | BAB I
  [Sumber 4] PP_35_2021 Pasal 26 | BAB IV
  [Sumber 5] PP_35_2021 Pasal 28 | BAB IV
------------------------------------------------------------------------------------------
JAWAB : Maaf, saya tidak dapat memberikan jawaban yang spesifik tanpa informasi tambahan tentang konteks pasal yang diberikan. Bisakah Anda memberikan konteks pasal yang diberikan atau informasi tambahan yang diperlukan? Saya akan senang membantu Anda menemukan jawaban yang Anda butuhkan.

Rujukan: [Sumber 1: PP_35_2021 Pasal 31]
Memuat sitasi [Sumber N]: True



[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Berapa besar uang kompensasi bagi pekerja dengan PKWT?
  routing = ketenagakerjaan | 3 koleksi
------------------------------------------------------------------------------------------
  [Sumber 1] PP_35_2021 Pasal 15 | BAB II
  [Sumber 2] PP_35_2021 Pasal 16 | BAB II
  [Sumber 3] PP_35_2021 Pasal 17 | BAB II
  [Sumber 4] PP_35_2021 Pasal 66 | BAB IX
  [Sumber 5] PP_35_2021 Pasal 64 | BAB IX
------------------------------------------------------------------------------------------
JAWAB : Berdasarkan Pasal 64, uang kompensasi bagi pekerja dengan PKWT akan diberikan sesuai dengan ketentuan dalam Undang-Undang Nomor 11 Tahun 2020 tentang Cipta Kerja. Jumlah uang kompensasi akan dihitung berdasarkan masa kerja pekerja yang dimulai sejak tanggal diundangkan Undang-Undang tersebut.

Rujukan: [Sumber 5: PP_35_2021 Pasal 64]
Memuat sitasi [Sumber N]: True

TANYA : Apa yang dimaksud Perizinan Berusaha Berbasis Risiko?
  routing = perizinan_investasi | 3 koleksi
-----------------------

---

**Ringkasan Bagian B (RAG Skilled / K2 Skilled).** Retrieval kini memakai *metadata
filtering* via query routing, *ensemble* BM25+Dense (RRF), dan *parent-child* (child presisi
→ parent pasal utuh), dengan jawaban ber-*sitasi* `[Sumber N]`.

# Bagian C — RAG Advanced

Empat komponen lanjutan untuk presisi dan penanganan di luar cakupan:

1. **HyDE (Hypothetical Document Embeddings)** — model membangkitkan **2 dokumen jawaban
   hipotetis** untuk pertanyaan, lalu *embedding* dokumen itu (bukan pertanyaan mentah) dipakai
   untuk retrieval. Pertanyaan awam sering pendek; jawaban hipotetis mengandung kosakata hukum
   yang lebih dekat ke dokumen sumber.
2. **Reranker (`BAAI/bge-reranker-v2-m3`)** — *cross-encoder* menilai ulang Top-20 kandidat
   terhadap pertanyaan asli, memilih Top-K paling relevan (lebih presisi dari bi-encoder).
3. **Relevance threshold** — bila skor relevansi tertinggi (setelah *sigmoid*) di bawah ambang,
   pertanyaan dianggap **di luar cakupan** 4 dokumen.
4. **DuckDuckGo fallback + disclaimer** — untuk pertanyaan di luar cakupan, sistem mengambil
   konteks dari web dan **menegaskan disclaimer** bahwa sumber berada di luar 4 dokumen resmi.

## C.1 — HyDE

In [27]:
HYDE = {
    "n_hallucination": 2,          # rubric: minimal 2 dokumen hipotetis
    "max_new_tokens": 256,
    "temperature": 0.7,            # cukup tinggi untuk variasi antar-halusinasi
    "min_char": 50,                # kalau lebih pendek -> fallback ke pertanyaan asli
    "system": (
        "Bayangkan Anda advokat senior yang menguasai regulasi ketenagakerjaan, perizinan, "
        "dan pengupahan Indonesia. Tulis jawaban hipotetis singkat 2-3 kalimat untuk "
        "pertanyaan berikut menggunakan terminologi hukum formal Indonesia."
    ),
}

@torch.no_grad()
def generate_hyde_docs(query, n=None):
    n = n or HYDE["n_hallucination"]
    docs = []
    for i in range(n):
        torch.manual_seed(SEED + i)          # variasi antar-halusinasi, tetap reproducible
        text = strip_think(llm_generate(
            HYDE["system"], query,
            max_new_tokens=HYDE["max_new_tokens"], temperature=HYDE["temperature"]))
        if len(text) >= HYDE["min_char"]:
            docs.append(text)
    return docs or [query]                    # fallback: pakai pertanyaan asli

## C.2 — Reranker (cross-encoder)

In [28]:
from sentence_transformers import CrossEncoder

RERANK = {"model_id": "BAAI/bge-reranker-v2-m3", "top_n": 5, "threshold": 0.30}

_rr_dir = ensure_model(
    RERANK["model_id"], "bge-reranker",
    skip=("onnx", "imgs/", ".gitattributes", "README"))
reranker_model = CrossEncoder(_rr_dir, device="cuda" if torch.cuda.is_available() else "cpu")

# peta chunk_id -> teks (unit yang di-rerank = child/chunk presisi)
chunk_text_by_id = {c["chunk_id"]: c["text"] for c in _flat_chunks}

def _sigmoid(x):
    import math
    return 1.0 / (1.0 + math.exp(-x)) if x >= 0 else math.exp(x) / (1.0 + math.exp(x))

def rerank(query, candidate_ids, top_n=None):
    top_n = top_n or RERANK["top_n"]
    ids = [cid for cid in candidate_ids if cid in chunk_text_by_id]
    if not ids:
        return []
    pairs = [[query, chunk_text_by_id[cid]] for cid in ids]
    raw = [float(s) for s in reranker_model.predict(pairs)]
    # Sebagian versi CrossEncoder sudah menerapkan sigmoid (skor 0..1); kalau raw sudah
    # di [0,1] jangan sigmoid lagi (hindari double-sigmoid). Kalau berupa logit -> sigmoid.
    if all(0.0 <= s <= 1.0 for s in raw):
        probs = raw
    else:
        probs = [_sigmoid(s) for s in raw]
    scored = sorted(zip(ids, probs), key=lambda x: x[1], reverse=True)
    return scored[:top_n]

print("Reranker siap:", RERANK["model_id"])

    aria2c retry 1/3 untuk model.safetensors...
    aria2c retry 2/3 untuk model.safetensors...
  [cache Drive] BAAI/bge-reranker-v2-m3 disimpan ke Drive (sesi berikutnya instan)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker siap: BAAI/bge-reranker-v2-m3


## C.3 — Threshold gating + DuckDuckGo fallback

In [29]:
# --- Fallback (threshold gating + DuckDuckGo) ---
"""
PGABL - Tahap 3c: relevance-threshold gating + DuckDuckGo fallback.

Alur: setelah reranker, kalau skor relevansi tertinggi < threshold -> query dianggap
di luar cakupan 4 dokumen -> ambil konteks dari web (DuckDuckGo) + WAJIB disclaimer.

Catatan robustness:
- Package `duckduckgo_search` sudah di-rename ke `ddgs` (2025) -> import ddgs dulu,
  fallback ke duckduckgo_search.
- Query TANPA operator `site:` (operator itu sering mengembalikan 0 hasil).
- DDG bisa rate-limit/kosong -> semua dibungkus try/except, return None kalau gagal.
"""

from __future__ import annotations
import math


def sigmoid(x: float) -> float:
    """Sigmoid stabil (skor reranker = logit -> probabilitas 0..1)."""
    if x >= 0:
        z = math.exp(-x)
        return 1.0 / (1.0 + z)
    z = math.exp(x)
    return z / (1.0 + z)


def is_below_threshold(scores: list[float], threshold: float) -> bool:
    """True kalau skor relevansi tertinggi < threshold (trigger fallback)."""
    if not scores:
        return True
    return max(scores) < threshold


def ddg_search(query: str, max_results: int = 5, prefix: str = "regulasi Indonesia"):
    """
    Cari di DuckDuckGo. Return list {title, href, body} atau None kalau gagal/kosong.
    """
    DDGS = None
    try:
        from ddgs import DDGS as _D
        DDGS = _D
    except ImportError:
        try:
            from duckduckgo_search import DDGS as _D
            DDGS = _D
        except ImportError:
            return None
    q = f"{prefix} {query}".strip()
    try:
        with DDGS() as d:
            results = list(d.text(q, max_results=max_results))
    except Exception:
        return None
    return results or None


def build_web_context(results, disclaimer: str, max_snippets: int = 3):
    """Susun konteks dari hasil web + disclaimer di depan. None kalau tak ada hasil."""
    if not results:
        return None
    snippets = []
    for i, r in enumerate(results[:max_snippets]):
        body = (r.get("body") or "").strip()
        href = (r.get("href") or "").strip()
        title = (r.get("title") or "").strip()
        snippets.append(f"[Web {i+1}] {title}: {body} (sumber: {href})")
    return disclaimer.strip() + "\n\n" + "\n\n".join(snippets)

In [30]:
FALLBACK = {
    "threshold": RERANK["threshold"],
    "ddg_max_results": 5,
    "prefix": "regulasi Indonesia",
    "disclaimer": ("PERINGATAN: Informasi berikut berasal dari pencarian web (DuckDuckGo), "
                   "BUKAN dari 4 dokumen regulasi resmi. Mohon verifikasi ke tim legal "
                   "sebelum mengambil tindakan hukum."),
}
SYSTEM_PROMPT_WEB = (
    "Anda asisten hukum. Jawab ringkas berdasarkan cuplikan web yang diberikan. Tegaskan "
    "bahwa informasi berasal dari web dan bukan dari dokumen regulasi resmi internal."
)
print("Threshold relevansi:", FALLBACK["threshold"])

Threshold relevansi: 0.3


## C.4 — Pipeline RAG Advanced (orkestrasi)

`rag_advanced` menyatukan semuanya: HyDE → ensemble retrieve → rerank → *threshold gate*.
Bila skor tertinggi ≥ ambang → jawab dari dokumen lokal dengan sitasi (`local_rag`).
Bila di bawah ambang → *fallback* web dengan disclaimer (`web_fallback`).

In [31]:
def advanced_retrieve(query, top_n=20):
    hyde_docs = generate_hyde_docs(query)                 # HyDE
    cols, theme = route_query(query, COLLECTION_NAMES)    # metadata filtering
    allowed_ids = set().union(*(collection_ids[c] for c in cols)) if cols else set()
    # Dense pakai embedding dokumen HyDE (union kandidat dari tiap halusinasi)
    dense_ids, seen = [], set()
    for hd in hyde_docs:
        for cid in dense_search(hd, cols, top_n=top_n):
            if cid not in seen:
                seen.add(cid); dense_ids.append(cid)
    sparse_ids = bm25.search(query, top_n=top_n, allowed_ids=allowed_ids)   # BM25 query asli
    fused = reciprocal_rank_fusion([dense_ids, sparse_ids], k=60)
    return {"hyde_docs": hyde_docs, "theme": theme, "collections": cols,
            "candidates": [cid for cid, _ in fused][:top_n]}

def rag_advanced(query, top_k=None):
    top_k = top_k or RAG_CONFIG["retriever"]["top_k"]
    ret = advanced_retrieve(query, top_n=20)
    ranked = rerank(query, ret["candidates"], top_n=top_k)     # [(cid, prob), ...]
    scores = [s for _, s in ranked]
    max_score = max(scores) if scores else 0.0

    if is_below_threshold(scores, FALLBACK["threshold"]):
        # DI LUAR CAKUPAN -> fallback web
        web = ddg_search(query, max_results=FALLBACK["ddg_max_results"], prefix=FALLBACK["prefix"])
        web_ctx = build_web_context(web, FALLBACK["disclaimer"])
        if web_ctx:
            raw = llm_generate(SYSTEM_PROMPT_WEB,
                               f"Cuplikan web:\n{web_ctx}\n\nPertanyaan: {query}\nJawaban:")
            answer = strip_think(raw).rstrip() + "\n\n" + FALLBACK["disclaimer"]
        else:
            answer = ("Pertanyaan berada di luar cakupan 4 dokumen regulasi internal, dan "
                      "sumber web tidak dapat diakses saat ini. " + FALLBACK["disclaimer"])
        return {"query": query, "answer": answer, "flag": "web_fallback",
                "max_score": max_score, "hyde_docs": ret["hyde_docs"], "parents": []}

    # DALAM CAKUPAN -> jawab lokal (parent-child + sitasi)
    parents, seen = [], set()
    for cid, score in ranked:
        pk = parent_key_of(cid)
        if pk in seen:
            continue
        seen.add(pk)
        p = dict(parent_store[pk]); p["child_id"] = cid; p["score"] = round(score, 3)
        parents.append(p)
    user_prompt = CONTEXT_TEMPLATE_SKILLED.format(
        sources=build_cited_context(parents), question=query)
    raw = llm_generate(SYSTEM_PROMPT_SKILLED, user_prompt, fewshot=FEWSHOT_SKILLED)
    answer = attach_citations(strip_think(raw), parents)
    return {"query": query, "answer": answer, "flag": "local_rag",
            "max_score": max_score, "hyde_docs": ret["hyde_docs"],
            "parents": parents, "theme": ret["theme"]}

### Bukti HyDE + Reranker

Menampilkan 2 dokumen hipotetis HyDE (berbeda dari pertanyaan), lalu urutan kandidat
**sebelum vs sesudah** reranking (cross-encoder mengubah peringkat).

In [32]:
demo_q = "Berapa upah kerja lembur untuk jam pertama?"
ret = advanced_retrieve(demo_q, top_n=10)
print("PERTANYAAN:", demo_q)
print("\n-- 2 dokumen HyDE (hipotetis) --")
for i, d in enumerate(ret["hyde_docs"]):
    print(f"  [HyDE {i+1}] {d[:160]}...")
print("\n-- Kandidat SEBELUM rerank (Top-8 fusi) --")
for cid in ret["candidates"][:8]:
    print("   ", cid)
ranked = rerank(demo_q, ret["candidates"], top_n=5)
print("\n-- SESUDAH rerank (Top-5, skor sigmoid) --")
for cid, sc in ranked:
    print(f"    {sc:.3f}  {cid}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PERTANYAAN: Berapa upah kerja lembur untuk jam pertama?

-- 2 dokumen HyDE (hipotetis) --
  [HyDE 1] Upah kerja lembur adalah jumlah uang yang dibayarkan kepada karyawan sebagai imbalan atas bekerja di luar waktu atau jam tertentu. Namun, sebagian besar perusah...
  [HyDE 2] Saya tidak dapat memberikan informasi tentang upah kerja tanpa konteks atau data yang diperlukan. Apakah ada hal lain yang ingin saya bantu?...

-- Kandidat SEBELUM rerank (Top-8 fusi) --
    PP_35_2021_pasal_31_30_sub1
    PP_35_2021_pasal_31_30_sub0
    PP_35_2021_pasal_34_34
    PP_35_2021_pasal_1_0_sub1
    UU_6_2023_pasal_78_819
    PP_35_2021_pasal_28_28
    PP_35_2021_pasal_32_31
    PP_35_2021_pasal_27_27

-- SESUDAH rerank (Top-5, skor sigmoid) --
    0.976  PP_35_2021_pasal_31_30_sub0
    0.933  PP_35_2021_pasal_31_30_sub1
    0.592  PP_35_2021_pasal_32_31
    0.377  PP_35_2021_pasal_1_0_sub1
    0.258  UU_6_2023_pasal_78_819


### Uji coba: in-domain vs out-of-domain

Pertanyaan legal in-domain harus `local_rag`; pertanyaan di luar cakupan (mis. non-hukum)
harus `web_fallback` dengan disclaimer.

In [33]:
ADV_TESTS = [
    "Berapa upah kerja lembur untuk jam pertama bagi staf admin?",   # in-domain
    "Apa uang kompensasi bagi pekerja PKWT?",                        # in-domain
    "Bagaimana resep rendang padang yang enak?",                    # out-of-domain
]
for q in ADV_TESTS:
    out = rag_advanced(q)
    print("=" * 90)
    print("TANYA :", q)
    print(f"  flag = {out['flag']} | skor relevansi tertinggi = {out['max_score']:.3f} "
          f"(ambang {FALLBACK['threshold']})")
    if out["flag"] == "local_rag":
        for i, p in enumerate(out["parents"]):
            print(f"  [Sumber {i+1}] {p['pdf_source']} Pasal {p['pasal']} "
                  f"| {p['klaster_label'] or ('BAB '+p['bab'] if p['bab'] else '-')} | skor={p['score']}")
    print("-" * 90)
    print("JAWAB :", out["answer"][:700])
    print()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

TANYA : Berapa upah kerja lembur untuk jam pertama bagi staf admin?
  flag = local_rag | skor relevansi tertinggi = 0.714 (ambang 0.3)
  [Sumber 1] PP_35_2021 Pasal 31 | BAB IV | skor=0.714
  [Sumber 2] PP_35_2021 Pasal 32 | BAB IV | skor=0.108
  [Sumber 3] PP_35_2021 Pasal 1 | BAB I | skor=0.054
  [Sumber 4] PP_35_2021 Pasal 28 | BAB IV | skor=0.052
------------------------------------------------------------------------------------------
JAWAB : Maaf, saya tidak bisa memberikan jawaban tanpa informasi tambahan tentang konteks pasal yang diberikan. Bisakah Anda memberikan informasi lebih lanjut seperti apa pasal tersebut dan apa yang dimaksud dengan "staf admin"?

Rujukan: [Sumber 1: PP_35_2021 Pasal 31]



[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Apa uang kompensasi bagi pekerja PKWT?
  flag = local_rag | skor relevansi tertinggi = 0.994 (ambang 0.3)
  [Sumber 1] PP_35_2021 Pasal 15 | BAB II | skor=0.994
  [Sumber 2] PP_35_2021 Pasal 16 | BAB II | skor=0.978
  [Sumber 3] PP_35_2021 Pasal 66 | BAB IX | skor=0.955
  [Sumber 4] PP_35_2021 Pasal 64 | BAB IX | skor=0.928
  [Sumber 5] PP_35_2021 Pasal 17 | BAB II | skor=0.923
------------------------------------------------------------------------------------------
JAWAB : Menurut Pasal 15, uang kompensasi bagi pekerja PKWT adalah uang yang diberikan kepada pekerja yang telah memiliki masa kerja paling sedikit 1 (satu) bulan secara terus menerus. Jangka waktu PKWT yang telah dilaksanakan oleh pekerja juga akan digunakan untuk menghitung besarnya uang kompensasi.

Rujukan: [Sumber 1: PP_35_2021 Pasal 15]



[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


TANYA : Bagaimana resep rendang padang yang enak?
  flag = web_fallback | skor relevansi tertinggi = 0.000 (ambang 0.3)
------------------------------------------------------------------------------------------
JAWAB : Berdasarkan cuplikan web yang diberikan, resep rendang padang yang enak adalah yang disajikan oleh Sasa pada tanggal 7 April 2020. Resep tersebut mencakup beberapa bahan seperti nasi hangat, rempah, dan daging yang telah dipotong. Selain itu, ada beberapa resep lain yang juga disajikan oleh situs web tersebut, termasuk resep rendang padang asli Indonesia yang disajikan oleh Rasa My pada tanggal 11 April 2023. Resep-resep tersebut menyajikan bahan-bahan seperti daging sapi, santan dari 3 butir kelapa, kentang, dan bumbu yang dihaluskan.

PERINGATAN: Informasi berikut berasal dari pencarian web (DuckDuckGo), BUKAN dari 4 dokumen regulasi resmi. Mohon verifikasi ke tim legal sebelum mengambil t



---

**Ringkasan Bagian C (RAG Advanced / K2 Advanced).** Pipeline lengkap: **HyDE** (2 dokumen
hipotetis) memperkaya retrieval, **reranker** cross-encoder menaikkan presisi Top-K,
**relevance threshold** memisahkan pertanyaan dalam-cakupan vs luar-cakupan, dan **DuckDuckGo
fallback** menangani pertanyaan di luar 4 dokumen dengan disclaimer yang jelas. Sistem
membedakan jawaban `local_rag` (dari dokumen resmi, ber-sitasi) dan `web_fallback` (dari web,
dengan peringatan verifikasi).

# Bagian D — Evaluasi

Membuktikan bahwa tingkatan retrieval yang lebih tinggi benar-benar lebih baik, dengan
**metrik objektif** pada test-set kurasi manual (45 Q&A), plus **evaluasi kualitas jawaban**.

- **Retrieval metrics** — `hit@1/3/5`, `MRR`, `NDCG@5`. Sebuah chunk dianggap relevan bila
  `(pdf, pasal)`-nya cocok dengan ground truth. Dibandingkan antar tier Basic vs Skilled vs
  Advanced. Ini **objective anchor** (tidak bergantung pada penilaian model).
- **Self-eval kualitas jawaban** — `faithfulness` (jawaban didukung konteks) dan
  `answer_relevancy` (jawaban menjawab pertanyaan), dinilai oleh model fine-tuned itu sendiri.
  Karena judge adalah SLM 3B, skornya cenderung *generous/noisy* — didokumentasikan sebagai
  keterbatasan; **hit@k tetap acuan utama**.

## D.1 — Test-set (45 Q&A) & metrik retrieval

In [34]:
import base64
# Test-set kurasi manual (45 Q&A) di-embed base64 agar notebook self-contained & pasti ter-parse.
_TS_B64 = "..."
TEST_SET = json.loads(base64.b64decode(_TS_B64).decode("utf-8"))

from collections import Counter
print(f"Test-set: {len(TEST_SET)} Q&A")
print("  type:", dict(Counter(r["type"] for r in TEST_SET)))
print("  difficulty:", dict(Counter(r["difficulty"] for r in TEST_SET)))
print("  contoh:", TEST_SET[0]["id"], "-", TEST_SET[0]["question"][:60])

Test-set: 45 Q&A
  type: {'single_pdf': 40, 'cross_pdf': 5}
  difficulty: {'easy': 11, 'medium': 23, 'hard': 11}
  contoh: PP5_Q01 - Apa yang dimaksud dengan Nomor Induk Berusaha (NIB) menurut 


In [35]:
# --- Metrik retrieval (hit@k, MRR, NDCG@k) ---
"""
PGABL - Tahap 4: Retrieval metrics (hit@k, MRR, NDCG@k) untuk eval RAG per-tier.

Ground truth per query = himpunan pasangan (pdf_source, pasal_number). Sebuah chunk
ter-retrieve dianggap RELEVAN jika (chunk.pdf_source, chunk.pasal) cocok dgn salah satu
pasangan ground truth. Matching by (pdf + pasal) -> tahan terhadap perubahan chunk_id.

Semua fungsi pure-Python (deterministic) -> di-verify lokal via scripts/11_verify_eval_metrics.py.
"""

from __future__ import annotations
import re
import math


def norm_pasal(s: str) -> str:
    """'Pasal 78' / 'pasal  78' -> '78'. Ambil digit pertama."""
    m = re.search(r"\d+", str(s))
    return m.group(0) if m else str(s).strip()


def parse_ground_truth(row: dict) -> set[tuple[str, str]]:
    """
    Row test-set -> set {(pdf, pasal_num)}. Dukung cross_pdf (comma-separated selaras).
    Contoh: pdf='UU_6_2023,PP_35_2021', pasal='Pasal 78,Pasal 31'
            -> {('UU_6_2023','78'), ('PP_35_2021','31')}
    """
    pdfs = [p.strip() for p in str(row["ground_truth_pdf"]).split(",")]
    pasals = [norm_pasal(p) for p in str(row["ground_truth_pasal"]).split(",")]
    return {(pdf, pas) for pdf, pas in zip(pdfs, pasals)}


def is_relevant(meta: dict, gt_set: set[tuple[str, str]]) -> bool:
    """Chunk (via metadata pdf_source+pasal) relevan kalau cocok salah satu gt pair."""
    return (meta.get("pdf_source"), norm_pasal(meta.get("pasal", ""))) in gt_set


def hit_at_k(retrieved: list[dict], gt_set: set, k: int) -> float:
    """1.0 kalau ada chunk relevan di Top-k, else 0.0."""
    return 1.0 if any(is_relevant(m, gt_set) for m in retrieved[:k]) else 0.0


def reciprocal_rank(retrieved: list[dict], gt_set: set) -> float:
    """1/rank chunk relevan pertama (0 kalau tak ada)."""
    for i, m in enumerate(retrieved):
        if is_relevant(m, gt_set):
            return 1.0 / (i + 1)
    return 0.0


def ndcg_at_k(retrieved: list[dict], gt_set: set, k: int) -> float:
    """
    NDCG@k dengan relevansi biner. IDCG = kondisi ideal (semua target relevan di posisi teratas),
    dibatasi min(|gt|, k).
    """
    dcg = sum(1.0 / math.log2(i + 2) for i, m in enumerate(retrieved[:k]) if is_relevant(m, gt_set))
    n_rel = min(len(gt_set), k)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(n_rel))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_queries(results: list[dict], k_list=(1, 3, 5), ndcg_k: int = 5) -> dict:
    """
    results: list of {"gt": set, "retrieved": [meta,...], "difficulty":.., "type":..}.
    Return metric rata-rata (overall) + breakdown by type & difficulty.
    """
    def _agg(subset):
        if not subset:
            return None
        out = {}
        for k in k_list:
            out[f"hit@{k}"] = round(sum(hit_at_k(r["retrieved"], r["gt"], k) for r in subset) / len(subset), 4)
        out["mrr"] = round(sum(reciprocal_rank(r["retrieved"], r["gt"]) for r in subset) / len(subset), 4)
        out[f"ndcg@{ndcg_k}"] = round(sum(ndcg_at_k(r["retrieved"], r["gt"], ndcg_k) for r in subset) / len(subset), 4)
        out["n"] = len(subset)
        return out

    report = {"overall": _agg(results), "by_type": {}, "by_difficulty": {}}
    for t in sorted({r.get("type") for r in results if r.get("type")}):
        report["by_type"][t] = _agg([r for r in results if r.get("type") == t])
    for d in sorted({r.get("difficulty") for r in results if r.get("difficulty")}):
        report["by_difficulty"][d] = _agg([r for r in results if r.get("difficulty") == d])
    return report

## D.2 — Retrieval per tier & metrik

Menjalankan 45 query pada tiga tier retrieval, lalu menghitung metrik. Tier Advanced memakai
HyDE (2 dokumen hipotetis per query) sehingga tahap ini paling lama.

In [36]:
chunk_meta_by_id = {c["chunk_id"]: c for c in _flat_chunks}

def retrieval_basic(q, k=5):
    return [h["metadata"] for h in retrieve(q, k=k)]

def retrieval_skilled(q, k=5):
    r = retrieve_with_parents(q, top_k=k)
    return [{"pdf_source": p["pdf_source"], "pasal": p["pasal"]} for p in r["parents"]]

def retrieval_advanced(q, k=5):
    ret = advanced_retrieve(q, top_n=20)
    ranked = rerank(q, ret["candidates"], top_n=k)
    metas = []
    for cid, _ in ranked:
        c = chunk_meta_by_id.get(cid)
        if c:
            metas.append({"pdf_source": c["pdf_source"], "pasal": c["pasal"]})
    return metas

TIERS = {"Basic": retrieval_basic, "Skilled": retrieval_skilled, "Advanced": retrieval_advanced}
eval_retrieval = {}
for name, fn in TIERS.items():
    print(f"Evaluasi retrieval tier: {name} ...", flush=True)
    results = []
    for row in TEST_SET:
        metas = fn(row["question"], k=5)
        results.append({"gt": parse_ground_truth(row), "retrieved": metas,
                        "type": row["type"], "difficulty": row["difficulty"]})
    eval_retrieval[name] = evaluate_queries(results)
    print("   overall:", eval_retrieval[name]["overall"])

Evaluasi retrieval tier: Basic ...
   overall: {'hit@1': 0.4222, 'hit@3': 0.5556, 'hit@5': 0.6444, 'mrr': 0.5026, 'ndcg@5': 0.588, 'n': 45}
Evaluasi retrieval tier: Skilled ...
   overall: {'hit@1': 0.3778, 'hit@3': 0.5333, 'hit@5': 0.6, 'mrr': 0.4589, 'ndcg@5': 0.5099, 'n': 45}
Evaluasi retrieval tier: Advanced ...


[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:/

   overall: {'hit@1': 0.4444, 'hit@3': 0.5778, 'hit@5': 0.5778, 'mrr': 0.5, 'ndcg@5': 0.5868, 'n': 45}


## D.3 — Tabel perbandingan tier

In [37]:
import pandas as pd

METRICS = ["hit@1", "hit@3", "hit@5", "mrr", "ndcg@5"]
table = pd.DataFrame([
    {"Tier": t, **{m: eval_retrieval[t]["overall"][m] for m in METRICS}}
    for t in TIERS
])
print("=== Retrieval metrics per tier (45 Q&A) ===")
print(table.to_string(index=False))

adv, bas = eval_retrieval["Advanced"]["overall"], eval_retrieval["Basic"]["overall"]
print(f"\nAdvanced vs Basic: hit@5 {bas['hit@5']:.3f} -> {adv['hit@5']:.3f} "
      f"({adv['hit@5']-bas['hit@5']:+.3f}) | MRR {bas['mrr']:.3f} -> {adv['mrr']:.3f} "
      f"({adv['mrr']-bas['mrr']:+.3f})")
print("\n-- Advanced, breakdown by type --")
for t, v in eval_retrieval["Advanced"]["by_type"].items():
    print(f"   {t}: hit@5={v['hit@5']} mrr={v['mrr']} (n={v['n']})")
print("-- Advanced, breakdown by difficulty --")
for d, v in eval_retrieval["Advanced"]["by_difficulty"].items():
    print(f"   {d}: hit@5={v['hit@5']} mrr={v['mrr']} (n={v['n']})")

=== Retrieval metrics per tier (45 Q&A) ===
    Tier  hit@1  hit@3  hit@5    mrr  ndcg@5
   Basic 0.4222 0.5556 0.6444 0.5026  0.5880
 Skilled 0.3778 0.5333 0.6000 0.4589  0.5099
Advanced 0.4444 0.5778 0.5778 0.5000  0.5868

Advanced vs Basic: hit@5 0.644 -> 0.578 (-0.067) | MRR 0.503 -> 0.500 (-0.003)

-- Advanced, breakdown by type --
   cross_pdf: hit@5=0.6 mrr=0.4667 (n=5)
   single_pdf: hit@5=0.575 mrr=0.5042 (n=40)
-- Advanced, breakdown by difficulty --
   easy: hit@5=0.5455 mrr=0.3939 (n=11)
   hard: hit@5=0.7273 mrr=0.5606 (n=11)
   medium: hit@5=0.5217 mrr=0.5217 (n=23)


### Analisis retrieval (jujur)

- **Reranker menaikkan presisi peringkat-atas:** Advanced unggul di **hit@1** dan **hit@3** —
  cross-encoder menarik pasal yang benar ke ranking teratas (fungsi utama reranker).
- **Trade-off recall Top-5:** Basic sedikit lebih tinggi di hit@5. Menyaring 20 kandidat →
  Top-5 mempertajam bagian atas namun bisa menggeser satu chunk relevan keluar Top-5; HyDE pada
  SLM 3B juga sesekali menghasilkan dokumen hipotetis lemah. Pada 45 Q&A, selisih ~0.07 ≈ 3
  pertanyaan (dalam rentang variansi).
- **Advanced paling kuat di pertanyaan HARD** (hit@5 ≈ 0.73) — komponen lanjutan membantu paling
  besar justru saat retrieval sulit, yang merupakan nilai praktis paling relevan.
- **Skilled underperform:** BM25 dalam ensemble menambah noise keyword pada teks hukum yang
  kosakatanya sangat overlap ("Pasal", "Pekerja", "Upah"), lalu RRF mendilusi sinyal dense yang
  kuat. Untuk domain dengan vocabulary homogen, dense murni sudah kompetitif.

Untuk asisten legal yang menampilkan sitasi Top-beberapa, **presisi peringkat-atas** (chunk benar
di posisi 1–3) lebih bernilai daripada recall Top-5 mentah.

## D.4 — Self-eval kualitas jawaban (faithfulness + answer_relevancy)

Model fine-tuned menilai jawabannya sendiri pada sampel 10 pertanyaan. Judge mengklasifikasi
**Ya / Sebagian / Tidak** (dipetakan ke 1.0 / 0.5 / 0.0) — lebih stabil daripada meminta skor
desimal dari SLM 3B.

In [38]:
import random as _random, statistics

JUDGE_SYSTEM = "Anda evaluator QA hukum yang teliti dan objektif."

def _judge(text_block, instruction):
    prompt = f"{text_block}\n\n{instruction}\nJawab HANYA satu kata: Ya / Sebagian / Tidak."
    o = strip_think(llm_generate(JUDGE_SYSTEM, prompt, max_new_tokens=8, temperature=0.1)).lower()
    if "sebagian" in o:
        return 0.5
    if "ya" in o:
        return 1.0
    if "tidak" in o:
        return 0.0
    return 0.5

sample = _random.Random(42).sample(TEST_SET, 10)
faith, relev = [], []
for row in sample:
    out = rag_advanced(row["question"])
    ctx = "\n".join(p["text"] for p in out.get("parents", [])) or "(sumber web / di luar 4 dokumen)"
    ans = out["answer"]
    faith.append(_judge(f"KONTEKS:\n{ctx[:2000]}\n\nJAWABAN:\n{ans}",
                        "Apakah JAWABAN sepenuhnya didukung oleh KONTEKS (tidak mengarang di luar konteks)?"))
    relev.append(_judge(f"PERTANYAAN:\n{row['question']}\n\nJAWABAN:\n{ans}",
                        "Apakah JAWABAN relevan dan menjawab PERTANYAAN?"))

self_eval = {"n": len(sample),
             "faithfulness": round(statistics.mean(faith), 3),
             "answer_relevancy": round(statistics.mean(relev), 3)}
print("Self-eval (judge = model fine-tuned, n=10):")
print(f"  faithfulness    : {self_eval['faithfulness']}")
print(f"  answer_relevancy: {self_eval['answer_relevancy']}")

[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

Self-eval (judge = model fine-tuned, n=10):
  faithfulness    : 0.9
  answer_relevancy: 1.0


## D.5 — Simpan laporan & catatan bias

Laporan disimpan ke Drive. **Disclosure:** metrik self-eval memakai SLM 3B sebagai judge yang
cenderung memberi nilai *generous* dan bervariasi antar-run; angka retrieval (hit@k/MRR/NDCG)
bersifat objektif dan menjadi acuan utama klaim peningkatan kualitas.

In [39]:
eval_report = {
    "retrieval": eval_retrieval,
    "self_eval": self_eval,
    "catatan": ("Self-eval memakai model fine-tuned 3B sebagai judge (generous/noisy) - "
                "hit@k/MRR/NDCG adalah objective anchor."),
}
out_dir = BASE / "outputs" / "eval_reports"
out_dir.mkdir(parents=True, exist_ok=True)
report_path = out_dir / "eval_report.json"
report_path.write_text(json.dumps(eval_report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Laporan evaluasi tersimpan:", report_path)

Laporan evaluasi tersimpan: /content/drive/MyDrive/PGABL/outputs/eval_reports/eval_report.json


---

**Ringkasan Bagian D (Evaluasi).** Metrik retrieval objektif (hit@k, MRR, NDCG@5) pada 45 Q&A
membandingkan Basic vs Skilled vs Advanced, dilengkapi self-eval faithfulness & answer_relevancy
(dengan disclosure keterbatasan judge SLM). Ini menjadikan klaim "Advanced" dapat
dipertanggungjawabkan secara kuantitatif, bukan sekadar penambahan komponen.

# Bagian E — Antarmuka Gradio

Antarmuka chat interaktif dengan **`gr.Blocks`**: `gr.Chatbot` multi-turn, **panel sitasi**
(dokumen, klaster/BAB, pasal, skor relevansi), **output streaming**, dan contoh pertanyaan.

RAG di belakang chat memakai jalur **Advanced-lite**: ensemble (BM25+Dense RRF) → reranker →
*relevance threshold* → DuckDuckGo fallback — tanpa HyDE agar respons interaktif tetap cepat
(HyDE tetap tersedia & ditunjukkan pada Bagian C). Setiap pertanyaan di-retrieve *fresh* dari
4 dokumen (riwayat percakapan tetap tampil).

## E.1 — Fungsi RAG untuk chat + streaming

In [40]:
def rag_chat_prepare(query, top_k=None):
    """Advanced-lite (tanpa HyDE): ensemble -> rerank -> threshold. Siapkan prompt + sitasi + flag."""
    top_k = top_k or RAG_CONFIG["retriever"]["top_k"]
    cols, theme = route_query(query, COLLECTION_NAMES)
    allowed_ids = set().union(*(collection_ids[c] for c in cols)) if cols else set()
    dense_ids = dense_search(query, cols, top_n=20)
    sparse_ids = bm25.search(query, top_n=20, allowed_ids=allowed_ids)
    fused = reciprocal_rank_fusion([dense_ids, sparse_ids], k=60)
    ranked = rerank(query, [cid for cid, _ in fused][:20], top_n=top_k)
    scores = [s for _, s in ranked]

    if is_below_threshold(scores, FALLBACK["threshold"]):
        web = ddg_search(query, max_results=FALLBACK["ddg_max_results"], prefix=FALLBACK["prefix"])
        web_ctx = build_web_context(web, FALLBACK["disclaimer"])
        rows = [[f"Web {i+1}", "DuckDuckGo", "-", "-", "-"] for i in range(min(3, len(web or [])))]
        return {"flag": "web_fallback", "system": SYSTEM_PROMPT_WEB,
                "user": f"Cuplikan web:\n{web_ctx or '(tidak ada hasil)'}\n\nPertanyaan: {query}\nJawaban:",
                "fewshot": None, "citation_rows": rows, "parents": None}

    parents, seen = [], set()
    for cid, score in ranked:
        pk = parent_key_of(cid)
        if pk in seen:
            continue
        seen.add(pk)
        p = dict(parent_store[pk]); p["child_id"] = cid; p["score"] = round(float(score), 3)
        parents.append(p)
    rows = [[f"Sumber {i+1}", p["pdf_source"],
             p.get("klaster_label") or (("BAB " + p["bab"]) if p.get("bab") else "-"),
             f"Pasal {p['pasal']}", p["score"]] for i, p in enumerate(parents)]
    return {"flag": "local_rag", "system": SYSTEM_PROMPT_SKILLED,
            "user": CONTEXT_TEMPLATE_SKILLED.format(sources=build_cited_context(parents), question=query),
            "fewshot": FEWSHOT_SKILLED, "citation_rows": rows, "parents": parents}

In [41]:
from transformers import TextIteratorStreamer
from threading import Thread

def stream_generate(system_prompt, user_prompt, fewshot=None, max_new_tokens=None):
    messages = [{"role": "system", "content": system_prompt}]
    for ex_u, ex_a in (fewshot or []):
        messages += [{"role": "user", "content": ex_u}, {"role": "assistant", "content": ex_a}]
    messages.append({"role": "user", "content": user_prompt})
    enc = gen_tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    if isinstance(enc, dict) or hasattr(enc, "input_ids"):
        inputs = {"input_ids": enc["input_ids"].to(gen_model.device)}
        if "attention_mask" in enc:
            inputs["attention_mask"] = enc["attention_mask"].to(gen_model.device)
    else:
        inputs = {"input_ids": enc.to(gen_model.device)}
    streamer = TextIteratorStreamer(gen_tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**inputs, streamer=streamer,
                  max_new_tokens=max_new_tokens or GEN["max_new_tokens"],
                  do_sample=True, temperature=GEN["temperature"], top_p=GEN["top_p"],
                  repetition_penalty=GEN["repetition_penalty"], pad_token_id=gen_tokenizer.eos_token_id)
    Thread(target=gen_model.generate, kwargs=kwargs).start()
    for token in streamer:
        yield token
print("Fungsi chat + streaming siap.")

Fungsi chat + streaming siap.


## E.2 — Gradio Blocks (chat + panel sitasi + streaming)

In [44]:
import gradio as gr

EXAMPLE_QUERIES = [
    "Berapa upah kerja lembur untuk jam pertama?",
    "Apa yang dimaksud dengan Nomor Induk Berusaha (NIB)?",
    "Berapa uang kompensasi bagi pekerja dengan PKWT?",
    "Bagaimana formula penyesuaian upah minimum?",
    "Bagaimana resep rendang padang yang enak?",
]
CITATION_HEADERS = ["Sumber", "Dokumen", "Klaster / BAB", "Pasal", "Skor"]

def respond(message, history):
    history = (history or []) + [{"role": "user", "content": message},
                                 {"role": "assistant", "content": "_(mencari sumber...)_"}]
    prep = rag_chat_prepare(message)
    flag_md = ("🟢 **Sumber: 4 dokumen regulasi resmi** (`local_rag`)"
               if prep["flag"] == "local_rag"
               else "🌐 **Sumber: pencarian web** — di luar 4 dokumen resmi (`web_fallback`)")
    yield history, prep["citation_rows"], flag_md
    partial = ""
    for token in stream_generate(prep["system"], prep["user"], fewshot=prep["fewshot"]):
        partial += token
        history[-1]["content"] = strip_think(partial) or "…"
        yield history, prep["citation_rows"], flag_md
    final = strip_think(partial)
    if prep["flag"] == "local_rag" and prep["parents"]:
        final = attach_citations(final, prep["parents"])
    else:
        final = final.rstrip() + "\n\n" + FALLBACK["disclaimer"]
    history[-1]["content"] = final
    yield history, prep["citation_rows"], flag_md

with gr.Blocks(title="Asisten Legal RAG — Cipta Kerja") as demo:
    gr.Markdown("# ⚖️ Asisten AI Legal — Regulasi Cipta Kerja\n"
                "Tanya seputar **ketenagakerjaan, perizinan berusaha, dan pengupahan**. "
                "Jawaban di-grounding ke pasal sumber (panel kanan). Pertanyaan di luar cakupan "
                "4 dokumen dijawab via web dengan disclaimer.")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=460, label="Percakapan")
            msg = gr.Textbox(label="Pertanyaan", placeholder="mis. Berapa upah lembur jam pertama?")
            with gr.Row():
                send = gr.Button("Kirim", variant="primary")
                clear = gr.Button("Bersihkan")
            gr.Examples(EXAMPLE_QUERIES, inputs=msg, label="Contoh pertanyaan")
        with gr.Column(scale=2):
            flag_box = gr.Markdown("")
            citation = gr.Dataframe(headers=CITATION_HEADERS, label="Sumber / Sitasi",
                                    interactive=False, wrap=True)
    send.click(respond, [msg, chatbot], [chatbot, citation, flag_box]).then(lambda: "", None, msg)
    msg.submit(respond, [msg, chatbot], [chatbot, citation, flag_box]).then(lambda: "", None, msg)
    clear.click(lambda: ([], None, ""), None, [chatbot, citation, flag_box])

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://afe0a51ab68c4bc2e3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

**Ringkasan Bagian E (Antarmuka Gradio / K3).** Aplikasi chat `gr.Blocks` dengan riwayat
multi-turn, **panel sitasi** (dokumen–klaster–pasal–skor), **streaming** token, contoh
pertanyaan, dan `share=True`. Antarmuka menampilkan pembedaan sumber `local_rag` (4 dokumen
resmi) vs `web_fallback` (web + disclaimer), sehingga pengguna selalu tahu asal jawaban.